# Bâtis de sociétaires impactés par les incendies — Gironde & Biscarrosse

**Demande métier** — *« Je souhaiterais savoir combien et positionner géographiquement
les bâtis (RP, RS, PNO) de nos sociétaires qui auraient effectivement été impactés
par les incendies. »*

**Livrable** — un fichier HTML autoportant (`livrables/carte_incendie_societaires.html`)
contenant les compteurs, la carte interactive et les tableaux, plus un export CSV
des contrats impactés pour les équipes de gestion.

---

## Méthode en une phrase

On croise les **points GPS des contrats habitation** (`contrat_mgar_gps_iris`) avec les
**emprises des bâtiments relevés dans la zone brûlée** par la cellule SIG, et on classe
chaque contrat par **niveau de certitude d'impact** selon sa distance au bâti le plus proche.

## Ce que contiennent les données incendie (analyse préalable)

| Couche | Fichier | Nb | CRS déclaré | Aire recalculée | Attributs |
|---|---|---|---|---|---|
| Contour feu Gironde | `2026_07_26_16h_Contour feu Gironde.shp` | 1 polygone | **EPSG:4326** | **37 020 ha** | `Surface` = 9702, `sup 2607` = 38502 |
| Bâti concerné Gironde | `2026_07_26_Bati_concerné_feu_Gironde.shp` | **1 607** | EPSG:2154 | 28,5 ha bâtis | `cleabs` *(inexploitable)*, `Surface` (m²) |
| Contour feu Biscarrosse | `2026_07_26_16hContour feu Biscarosse.shp` | 1 polygone | **absent (.prj manquant)** | **3 422 ha** | `FID` |
| Bâti concerné Biscarrosse | `2026_07_2026_Bati_concerné_feu_Biscarrosse.shp` | **1 060** | EPSG:2154 | 22,9 ha bâtis | `Surface` (m²) |

Points de vigilance relevés et traités dans ce notebook :

1. **CRS hétérogènes.** Les contours sont en WGS84 (Gironde) et sans projection déclarée
   (Biscarrosse). Les emprises de Biscarrosse sont bien du **Lambert 93** — vérifié par
   l'étendue des coordonnées (x ≈ 368 000, y ≈ 6 372 000) et par le recouvrement à 98 %
   avec les bâtis du même dossier. Le notebook force donc `EPSG:2154` pour ce fichier.
2. **`cleabs` est inexploitable** sur le fichier Gironde : la même valeur
   `BATIMENT0000000245412688` est répétée sur les 1 607 lignes (artefact de jointure).
   On génère donc notre propre identifiant de bâtiment (`bat_id`).
3. **Sur les couches de bâti, `Surface` est bien l'emprise au sol** en m² (contrôlé :
   `Surface` == aire géométrique calculée en Lambert 93). Minimum 50 m² sur les deux
   couches → les petites annexes (abris, cabanes) ont déjà été filtrées à la source.
   En revanche, sur le **contour** Gironde les attributs `Surface` (9 702) et
   `sup 2607` (38 502) ne concordent pas avec l'aire géométrique (37 020 ha) : ces
   deux champs ne sont pas repris, toutes les surfaces du notebook sont **recalculées**.
4. **Périmètre ≠ bâti concerné.** Le contour du feu couvre 370 km² (Gironde) et 34 km²
   (Biscarrosse), en grande majorité de la forêt. Répondre « impactés » par
   *point dans le périmètre* surestimerait massivement : la référence à retenir est le
   **bâti relevé dans l'emprise**, le périmètre ne servant qu'à qualifier l'exposition.

## 1. Paramètres

Tout ce qui se règle est ici. Les seuils de distance sont le seul vrai arbitrage :
ils absorbent l'imprécision du géocodage des adresses.

In [ ]:
from __future__ import annotations

import html as _html
import json
import warnings
from datetime import date
from pathlib import Path

import folium
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Point

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

# --- Arborescence ---------------------------------------------------------- #
# On part du répertoire courant et on remonte jusqu'à trouver `data_incendie`.
# Le notebook tourne ainsi aussi bien à plat (poste d'exploration) que depuis le
# sous-dossier incendie/ du dépôt, sans avoir à retoucher ce chemin.
RACINE = Path.cwd()
for _candidat in (RACINE, *RACINE.parents):
    if (_candidat / "data_incendie").is_dir():
        RACINE = _candidat
        break
DOSSIER_INCENDIE = RACINE / "data_incendie"
DOSSIER_SORTIE = RACINE / "livrables"
DOSSIER_SORTIE.mkdir(exist_ok=True)

FICHIER_HTML = DOSSIER_SORTIE / "carte_incendie_societaires.html"
FICHIER_CSV = DOSSIER_SORTIE / "contrats_impactes.csv"

# --- Les deux sinistres ---------------------------------------------------- #
# `crs_defaut` ne sert que si le shapefile n'embarque pas de .prj.
FEUX = {
    "Gironde":     {"dossier": "FEU GIRONDE",    "contour": "*Contour*.shp",
                    "bati": "*Bati*.shp", "crs_defaut": 2154, "debut": "2026-07-23"},
    "Biscarrosse": {"dossier": "FEU BISCAROSSE", "contour": "*Contour*.shp",
                    "bati": "*Bati*.shp", "crs_defaut": 2154, "debut": "2026-07-24"},
}
DATE_DEBUT_FEUX = min(f["debut"] for f in FEUX.values())
RELEVE_SIG = "26/07/2026 16 h"

# --- Géodésie -------------------------------------------------------------- #
CRS_METRIQUE = 2154        # Lambert 93 : toutes les distances sont calculées ici
CRS_AFFICHAGE = 4326       # WGS84 : projection de la carte

# --- Seuils d'appariement (mètres) ----------------------------------------- #
# Un point contrat tombe rarement pile dans l'emprise du bâtiment : le géocodage
# ramène souvent l'adresse au bord de la parcelle ou à l'axe de la voie.
SEUIL_TRES_PROBABLE_M = 10.0    # au-delà de l'emprise mais collé à un bâti de l'emprise
SEUIL_PROBABLE_M = 25.0         # tolérance haute du géocodage adresse
SEUILS_SENSIBILITE_M = [0.0, 5.0, 10.0, 25.0, 50.0, 100.0]

# Écart au-delà duquel plusieurs géocodages d'un même contrat ne désignent
# manifestement pas le même bâtiment : la position retenue devient un choix par
# défaut, à signaler plutôt qu'à présenter comme une localisation.
SEUIL_ECART_POSITIONS_M = 100.0

# Niveaux retenus comme « effectivement impacté » dans les compteurs de tête.
NIVEAUX_IMPACTES = ("Certain — dans l'emprise brûlée", "Très probable — à moins de 10 m", "Probable — à moins de 25 m")

# --- Précision du géocodage ------------------------------------------------ #
# Si la table de géocodage expose une colonne qualifiant la précision du point,
# elle fait foi et remplace l'heuristique multi-voies. Le notebook la cherche
# sous ces noms ; il suffit de l'ajouter au SELECT pour qu'elle soit exploitée.
CANDIDATS_COLONNE_PRECISION = (
    "level_contrat_mgar",          # nom réel dans contrat_mgar_gps_iris
    "type_geocodage", "result_type", "ban_result_type", "niveau_geocodage",
    "precision_geocodage", "qualite_geocodage", "code_precision_geocodage",
    "code_precision", "type_localisation", "niveau_localisation",
    "score_geocodage", "indice_confiance_geocodage",
)
# Modalités dénotant un point NON posé sur l'adresse. Compléter après lecture de
# REQ_GPS_MODALITES_PRECISION : les libellés maison ne sont pas ceux de la BAN.
MODALITES_CENTROIDE_COMMUNE = {
    "municipality", "commune", "centroide_commune", "centroide commune",
    "centroïde commune", "mairie", "city", "municipalite",
}
MODALITES_VOIE = {"street", "voie", "rue", "axe", "troncon", "tronçon"}
MODALITES_LIEU_DIT = {"locality", "lieu-dit", "lieu_dit", "hameau"}
MODALITES_ADRESSE = {"housenumber", "numero", "numéro", "adresse", "point_adresse",
                     "batiment", "bâtiment", "parcelle", "toit", "rooftop"}

# --- Ce que le niveau de géocodage autorise à conclure ---------------------- #
# `municipality` place le point au centroïde de la commune, `locality` au centre
# d'un lieu-dit : à cette échelle, la distance à un bâtiment ne veut rien dire.
# Ces contrats sont sortis du comptage « impactés » — les compter reviendrait à
# fabriquer des faux positifs par pur hasard géométrique — et suivis à part.
NIVEAUX_GEOCODAGE_INEXPLOITABLES = ("Centroïde commune", "Lieu-dit")
NIVEAU_INEXPLOITABLE = "Position trop imprécise pour conclure"

# `street` pose le point sur l'axe de la voie, pas sur le bâti : tomber dans une
# emprise y est une coïncidence géométrique, pas une preuve. On plafonne donc ces
# contrats à « très probable » plutôt que « certain ».
PLAFONNER_NIVEAU_VOIE = True

# --- Jointure géocodage × contrat ------------------------------------------ #
# True  : jointure sur 5 clés, adresse comprise — élimine les doublons dès le SQL,
#         mais écarte les contrats dont le libellé d'adresse diffère entre les
#         deux tables (mesuré : au moins 100 clés ont un point identique et un
#         libellé différent, et l'appariement perd 5,1 % des contrats).
# False : jointure sur 3 clés — recall maximal. Les contrats à positions multiples
#         sont dédoublonnés plus bas côté Python, en retenant la position la plus
#         proche d'un bâti de l'emprise, ce que le SQL ne sait pas faire.
JOINTURE_STRICTE_ADRESSE = False

# --- Segmentation demandée ------------------------------------------------- #
SEGMENTS_CIBLE = ("RP", "RS", "PNO")

# --- Carte ----------------------------------------------------------------- #
FOND_DE_CARTE = "OpenStreetMap"   # None => aucun fond tuilé (poste sans accès Internet)
# Plafond d'affichage des couches de contexte : elles servent de repère de volume,
# pas d'outil de gestion. Au-delà, l'affichage est échantillonné et le fait est
# annoncé — les comptages, eux, portent toujours sur la totalité.
MAX_POINTS_CONTEXTE = 20_000
# Leaflet est embarqué dans le dépôt (incendie/assets) et injecté en dur dans le HTML :
# le livrable ne dépend donc d'aucun CDN, seules les tuiles du fond de carte
# nécessitent un accès réseau. Mettre False pour repasser sur les CDN de folium.
DOSSIER_ASSETS = RACINE / "incendie" / "assets"
LEAFLET_EMBARQUE = all((DOSSIER_ASSETS / f).exists()
                       for f in ("leaflet.js", "leaflet.css", "jquery.js"))

DATE_ANALYSE = date.today().isoformat()
print("Racine projet :", RACINE)
print("Données incendie :", DOSSIER_INCENDIE, "—", DOSSIER_INCENDIE.exists())

## 2. Chargement et normalisation des données incendie

On ramène tout en Lambert 93, on force le CRS manquant de Biscarrosse et on
crée un identifiant de bâtiment fiable.

In [ ]:
def _lire_couche(chemin: Path, crs_defaut: int) -> gpd.GeoDataFrame:
    """Lit un shapefile et le ramène en Lambert 93, en réparant un CRS absent."""
    gdf = gpd.read_file(chemin)
    if gdf.crs is None:
        gdf = gdf.set_crs(crs_defaut)
        print(f"  ⚠️  {chemin.name} : aucun .prj → CRS forcé à EPSG:{crs_defaut}")
    return gdf.to_crs(CRS_METRIQUE)


contours_l, batis_l = [], []
for nom, cfg in FEUX.items():
    dossier = DOSSIER_INCENDIE / cfg["dossier"]
    f_contour = sorted(dossier.glob(cfg["contour"]))[0]
    f_bati = sorted(dossier.glob(cfg["bati"]))[0]
    print(f"• {nom}")

    contour = _lire_couche(f_contour, cfg["crs_defaut"])
    contour["feu"] = nom
    contour["fichier"] = f_contour.name
    contours_l.append(contour[["feu", "fichier", "geometry"]])

    bati = _lire_couche(f_bati, cfg["crs_defaut"])
    bati["feu"] = nom
    bati["fichier"] = f_bati.name
    bati["surface_bati_m2"] = bati.geometry.area.round(1)
    batis_l.append(bati[["feu", "fichier", "surface_bati_m2", "geometry"]])

contours = gpd.GeoDataFrame(pd.concat(contours_l, ignore_index=True),
                            geometry="geometry", crs=CRS_METRIQUE)
batis = gpd.GeoDataFrame(pd.concat(batis_l, ignore_index=True),
                         geometry="geometry", crs=CRS_METRIQUE)

# `cleabs` étant constant sur le fichier Gironde, on fabrique notre propre clé.
batis["bat_id"] = [f"{f[:4].upper()}-{i:05d}" for i, f in enumerate(batis["feu"], 1)]

contours["surface_feu_ha"] = (contours.geometry.area / 10_000).round(0)

print()
display(contours.drop(columns="geometry"))
display(
    batis.groupby("feu")
    .agg(nb_batis=("bat_id", "size"),
         surface_totale_m2=("surface_bati_m2", "sum"),
         surface_mediane_m2=("surface_bati_m2", "median"),
         surface_max_m2=("surface_bati_m2", "max"))
    .round(0)
)

In [ ]:
# Contrôle de cohérence : les bâtis relevés tombent-ils bien dans leur périmètre ?
for nom in FEUX:
    c = contours.loc[contours.feu == nom].geometry.union_all()
    b = batis.loc[batis.feu == nom]
    dedans = b.representative_point().within(c).mean()
    print(f"{nom:<12} {len(b):>5} bâtis — {dedans:6.1%} strictement dans le contour "
          f"(le reste affleure la limite du périmètre)")

## 3. Extraction des contrats habitation

### Deux sources de multiplicité, traitées dans l'ordre

**1. Plusieurs contrats par clé.** `contrat_mgar` porte plusieurs lignes par couple
`(id_societaire, numero_intercalaire)` : ce sont des **contrats successifs**, chacun
avec ses dates d'effet. On ne retient que le plus récent — `QUALIFY ROW_NUMBER()`
sur `date_effet DESC`, puis `date_premier_effet` et l'identifiant contrat pour
départager les ex aequo. C'est ce contrat qui décrit le risque assuré aujourd'hui,
donc c'est **lui qui fournit l'adresse**.

**2. Plusieurs géocodages par contrat.** Le contrat retenu pilote alors la jointure :
son code postal sélectionne la ligne de géocodage correspondante. Ce qui subsiste est
arbitré côté Python sur `level_contrat_mgar` — la requête n'a pas de critère pour le
faire, et le choix mérite d'être tracé.

L'ordre compte : dédoublonner le géocodage avant le contrat reviendrait à choisir une
position pour une adresse qui n'est peut-être plus celle du contrat en cours.

`date_resiliation IS NULL` restreint aux contrats en cours. Un contrat résilié entre
le départ du feu et l'extraction en sortirait — à surveiller si l'analyse est rejouée
plusieurs semaines après le sinistre.

Le reste de la requête :

* un **pré-filtre sur la bounding box des deux feux**, calculée depuis les shapefiles :
  inutile de rapatrier la France entière pour une analyse landaise/girondine ;
* quelques **colonnes facultatives** (`code_type_bien`, `surface_habitable`…) qui
  enrichissent le livrable. Le notebook fonctionne sans : il détecte les colonnes
  présentes et adapte tableaux et export ;
* `numero_contrat` (l'`id` de `contrat_mgar`) est ramené pour **tracer quel contrat a
  été retenu**, et n'est utilisé nulle part comme clé — un numéro distinct étant
  attribué aux dépendances, il n'en est pas une.

Attention en reprenant cette requête : `CONCAT(...) AS id` et `t2.id` produiraient
**deux colonnes nommées `id`**, ce que BigQuery rejette. D'où l'alias `numero_contrat`.

**`t1.geom` n'est pas sélectionné.** BigQuery interdit `SELECT DISTINCT` sur une
colonne `GEOGRAPHY` — le type n'est ni groupable ni comparable. La colonne est de
toute façon redondante avec `lon_contrat_mgar` / `lat_contrat_mgar`, à partir
desquels la géométrie est reconstruite côté Python. Si le WKT est vraiment
nécessaire, `ST_ASTEXT(t1.geom) AS geom_wkt` renvoie une `STRING`, qui supporte
le `DISTINCT`.

**La jointure porte sur 5 clés**, adresse et commune comprises. C'est plus strict,
donc plus sûr sur les doublons — mais un `INNER JOIN` sur des libellés d'adresse
écarte sans le dire les lignes dont le formatage diffère entre les deux tables. La
constante `REQUETE_CONTROLE_JOINTURE` chiffre cet écart : à passer une fois dans
BigQuery avant de communiquer les chiffres.

Le notebook s'exécute dans **trois modes**, dans cet ordre :
`BigQuery` → `fichier local` → `simulation` (jeu de test synthétique, pour valider
la chaîne de bout en bout sans accès aux données réelles).

In [ ]:
# Emprise des deux feux, élargie de 2 km, en WGS84 -> pré-filtre de la requête.
bbox = (contours.geometry.buffer(2_000).to_crs(CRS_AFFICHAGE).total_bounds)
LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = [round(v, 4) for v in bbox]
print(f"Bounding box de travail : lon [{LON_MIN}, {LON_MAX}] — lat [{LAT_MIN}, {LAT_MAX}]")

# Colonnes facultatives : elles enrichissent le livrable (nature du bien, commune,
# surface) et sont détectées automatiquement plus bas. Mettre "" pour s'en passer.
# Colonnes facultatives : elles enrichissent le livrable et sont détectées
# automatiquement plus bas. Mettre "" pour s'en passer. Ne pas y remettre
# commune_adresse_risque : elle est déjà prise sur t1, et BigQuery rejette les
# noms de colonnes dupliqués dans le résultat.
# Colonnes facultatives : elles enrichissent le livrable et sont détectées
# automatiquement plus bas. Vider le tuple pour s'en passer.
COLONNES_FACULTATIVES = (
    "code_type_bien",            # IM immeuble / MP maison / MH mobile home
    "surface_habitable",
    "nombre_pieces_totales",
    "montant_capital_mobilier",
)
_opt_cte = "".join(f",\n    {c}" for c in COLONNES_FACULTATIVES)
_opt_sel = "".join(f",\n  c.{c}" for c in COLONNES_FACULTATIVES)

_JOIN_ADRESSE = ("""
  AND t1.rue_adresse_risque         = c.rue_adresse_risque
  AND t1.commune_adresse_risque     = c.commune_adresse_risque"""
                 if JOINTURE_STRICTE_ADRESSE else
                 "\n  -- jointure sur 3 clés : voir JOINTURE_STRICTE_ADRESSE")

# --------------------------------------------------------------------------- #
# Deux sources de multiplicité, traitées dans cet ordre.
#
# 1. `contrat_mgar` porte plusieurs contrats par couple (id_societaire,
#    numero_intercalaire) : ce sont des contrats successifs, avec leurs propres
#    dates d'effet. On ne garde que le plus récent — c'est lui qui décrit le
#    risque assuré aujourd'hui, et donc lui qui fournit l'adresse.
#
# 2. La table de géocodage porte plusieurs lignes par contrat. Le contrat retenu
#    ci-dessus pilote la jointure : son code postal sélectionne le géocodage
#    correspondant. Ce qui subsiste est arbitré côté Python sur
#    `level_contrat_mgar`, la requête n'ayant pas de critère pour le faire.
#
# `t1.geom` est volontairement absent : BigQuery interdit SELECT DISTINCT sur une
# colonne GEOGRAPHY. La géométrie est reconstruite depuis lon/lat ; si le WKT est
# nécessaire, ST_ASTEXT(t1.geom) AS geom_wkt est une STRING et supporte DISTINCT.
# --------------------------------------------------------------------------- #
REQUETE_SQL = f"""
WITH contrat_courant AS (
  SELECT
    id_societaire,
    numero_intercalaire,
    id AS numero_contrat,             -- traçabilité : quel contrat a été retenu
    rue_adresse_risque,
    code_postal_adresse_risque,
    commune_adresse_risque,
    date_premier_effet,
    date_effet,
    code_sous_type,                   -- segmentation RP / RS / PNO
    code_qualite_assure_habitation{_opt_cte}
  FROM `matmut-dni-datalake-prd-8ec7.silver_produitetcontrat_sigmainframe_prd.contrat_mgar`
  WHERE tech_date_fin_historisation IS NULL
    AND date_resiliation IS NULL      -- contrats en cours uniquement
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY id_societaire, numero_intercalaire
    ORDER BY date_effet DESC NULLS LAST, date_premier_effet DESC NULLS LAST, id DESC
  ) = 1
)
SELECT DISTINCT
  c.id_societaire,
  c.numero_intercalaire,
  CONCAT(c.id_societaire, c.numero_intercalaire) AS id,
  c.numero_contrat,
  c.rue_adresse_risque,
  c.code_postal_adresse_risque,
  c.commune_adresse_risque,
  c.date_premier_effet,
  c.date_effet,
  c.code_sous_type,
  c.code_qualite_assure_habitation{_opt_sel},
  t1.lon_contrat_mgar,
  t1.lat_contrat_mgar,
  t1.level_contrat_mgar               -- housenumber / street / locality / municipality
FROM contrat_courant c
INNER JOIN `matmut-dda-irisation-prd-26f3.gold_irisation_prd.contrat_mgar_gps_iris` t1
  ON  t1.id_societaire              = c.id_societaire
  AND t1.numero_intercalaire        = c.numero_intercalaire
  AND t1.code_postal_adresse_risque = c.code_postal_adresse_risque{_JOIN_ADRESSE}
-- Pré-filtre sur l'emprise des deux feux : évite de rapatrier la France entière
WHERE t1.lon_contrat_mgar BETWEEN {LON_MIN} AND {LON_MAX}
  AND t1.lat_contrat_mgar BETWEEN {LAT_MIN} AND {LAT_MAX}
"""
print(REQUETE_SQL)

# --------------------------------------------------------------------------- #
# Contrôle à passer une fois dans BigQuery.
#
# La jointure porte sur 5 clés, dont `rue_adresse_risque` et `commune_adresse_risque`.
# Elles resserrent le résultat — c'est l'intention — mais un INNER JOIN sur des
# libellés d'adresse écarte silencieusement les lignes dont le formatage diffère
# entre la table gold géocodée (t1) et la source silver (t2) : casse, accents,
# abréviations, espaces. Ce contrôle chiffre l'écart avant de conclure.
# --------------------------------------------------------------------------- #
REQUETE_CONTROLE_JOINTURE = f"""
WITH base AS (
  SELECT t1.id_societaire, t1.numero_intercalaire, t1.code_postal_adresse_risque,
         t1.rue_adresse_risque, t1.commune_adresse_risque
  FROM `matmut-dda-irisation-prd-26f3.gold_irisation_prd.contrat_mgar_gps_iris` t1
  WHERE t1.lon_contrat_mgar BETWEEN {LON_MIN} AND {LON_MAX}
    AND t1.lat_contrat_mgar BETWEEN {LAT_MIN} AND {LAT_MAX}
)
SELECT
  COUNT(DISTINCT CONCAT(b.id_societaire, b.numero_intercalaire))          AS contrats_t1,
  COUNT(DISTINCT IF(j3.id_societaire IS NOT NULL,
        CONCAT(b.id_societaire, b.numero_intercalaire), NULL))            AS apparies_3_cles,
  COUNT(DISTINCT IF(j5.id_societaire IS NOT NULL,
        CONCAT(b.id_societaire, b.numero_intercalaire), NULL))            AS apparies_5_cles
FROM base b
LEFT JOIN `matmut-dni-datalake-prd-8ec7.silver_produitetcontrat_sigmainframe_prd.contrat_mgar` j3
  ON  b.id_societaire = j3.id_societaire
  AND b.numero_intercalaire = j3.numero_intercalaire
  AND b.code_postal_adresse_risque = j3.code_postal_adresse_risque
  AND j3.tech_date_fin_historisation IS NULL
LEFT JOIN `matmut-dni-datalake-prd-8ec7.silver_produitetcontrat_sigmainframe_prd.contrat_mgar` j5
  ON  b.id_societaire = j5.id_societaire
  AND b.numero_intercalaire = j5.numero_intercalaire
  AND b.code_postal_adresse_risque = j5.code_postal_adresse_risque
  AND b.rue_adresse_risque = j5.rue_adresse_risque
  AND b.commune_adresse_risque = j5.commune_adresse_risque
  AND j5.tech_date_fin_historisation IS NULL
"""
print("\n--- Contrôle de jointure (à exécuter une fois dans BigQuery) ---")
print("Si apparies_5_cles < apparies_3_cles, l'écart correspond à des contrats perdus")
print("sur un écart de libellé d'adresse, pas à des contrats réellement absents.")


# --------------------------------------------------------------------------- #
# La table de géocodage porte-t-elle elle-même des doublons ?
#
# Si `contrat_mgar_gps_iris` contient plusieurs lignes pour un même contrat, la
# jointure les propage : le doublon ne vient alors pas de contrat_mgar. Ces trois
# requêtes se passent en cascade, de la volumétrie à l'exemple concret.
# --------------------------------------------------------------------------- #
TABLE_GPS = "matmut-dda-irisation-prd-26f3.gold_irisation_prd.contrat_mgar_gps_iris"
_BBOX = (f"lon_contrat_mgar BETWEEN {LON_MIN} AND {LON_MAX}\n"
         f"    AND lat_contrat_mgar BETWEEN {LAT_MIN} AND {LAT_MAX}")

# 1. Quelles colonnes existent ? (utile pour repérer une colonne d'historisation)
REQ_GPS_COLONNES = """
SELECT column_name, data_type
FROM `matmut-dda-irisation-prd-26f3.gold_irisation_prd.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'contrat_mgar_gps_iris'
ORDER BY ordinal_position
"""

# 1 bis. Repérer la colonne qui qualifie la précision du géocodage : elle dit si le
# point est posé sur le numéro, sur la voie, ou sur le centroïde de la commune.
# C'est la source qui fait foi — bien supérieure à l'heuristique multi-voies.
REQ_GPS_COLONNE_PRECISION = """
SELECT column_name, data_type
FROM `matmut-dda-irisation-prd-26f3.gold_irisation_prd.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'contrat_mgar_gps_iris'
  AND REGEXP_CONTAINS(LOWER(column_name),
      r'type|score|precis|qualit|niveau|geocod|ban|result|centro|match|fiab|confian|localis')
ORDER BY ordinal_position
"""

# 1 ter. Une fois la colonne identifiée, en lire les modalités. Remplacer <COLONNE>.
REQ_GPS_MODALITES_PRECISION = f"""
SELECT
  <COLONNE>                                                  AS modalite,
  COUNT(*)                                                   AS lignes,
  COUNT(DISTINCT CONCAT(id_societaire, numero_intercalaire))  AS contrats
FROM `{TABLE_GPS}`
WHERE {_BBOX}
GROUP BY modalite
ORDER BY contrats DESC
"""

# 2. Volumétrie : à quel niveau de clé l'unicité est-elle atteinte ?
REQ_GPS_UNICITE = f"""
SELECT
  COUNT(*)                                                   AS lignes,
  COUNT(DISTINCT CONCAT(id_societaire, numero_intercalaire))  AS cles_soc_intercalaire,
  COUNT(DISTINCT CONCAT(id_societaire, numero_intercalaire,
                        code_postal_adresse_risque))          AS cles_avec_code_postal,
  COUNT(DISTINCT CONCAT(id_societaire, numero_intercalaire,
                        code_postal_adresse_risque,
                        IFNULL(rue_adresse_risque, ''),
                        IFNULL(commune_adresse_risque, '')))   AS cles_5_colonnes
FROM `{TABLE_GPS}`
WHERE {_BBOX}
"""

# 3. Sur les clés en doublon, qu'est-ce qui varie exactement ?
REQ_GPS_CAUSES = f"""
WITH doublons AS (
  SELECT
    id_societaire,
    numero_intercalaire,
    COUNT(*)                                          AS n_lignes,
    COUNT(DISTINCT code_postal_adresse_risque)        AS n_code_postal,
    COUNT(DISTINCT rue_adresse_risque)                AS n_rue,
    COUNT(DISTINCT commune_adresse_risque)            AS n_commune,
    COUNT(DISTINCT FORMAT('%.6f|%.6f',
          lon_contrat_mgar, lat_contrat_mgar))         AS n_positions
  FROM `{TABLE_GPS}`
  WHERE {_BBOX}
  GROUP BY id_societaire, numero_intercalaire
  HAVING COUNT(*) > 1
)
SELECT
  COUNT(*)                     AS cles_en_doublon,
  SUM(n_lignes - 1)            AS lignes_en_exces,
  COUNTIF(n_code_postal > 1)   AS varie_code_postal,
  COUNTIF(n_rue > 1)           AS varie_rue,
  COUNTIF(n_commune > 1)       AS varie_commune,
  COUNTIF(n_positions > 1)     AS varie_position,
  COUNTIF(n_code_postal = 1 AND n_rue = 1
          AND n_commune = 1 AND n_positions = 1) AS lignes_identiques
FROM doublons
"""

# 4. Les cas les plus chargés, pour aller voir sur pièce.
REQ_GPS_EXEMPLES = f"""
SELECT *
FROM `{TABLE_GPS}`
WHERE {_BBOX}
  AND CONCAT(id_societaire, numero_intercalaire) IN (
    SELECT CONCAT(id_societaire, numero_intercalaire)
    FROM `{TABLE_GPS}`
    WHERE {_BBOX}
    GROUP BY id_societaire, numero_intercalaire
    HAVING COUNT(*) > 1
    ORDER BY COUNT(*) DESC
    LIMIT 5
  )
ORDER BY id_societaire, numero_intercalaire
"""

print("\n" + "=" * 74)
print("Doublons dans la table de géocodage — 4 requêtes à passer dans BigQuery")
print("=" * 74)
print("REQ_GPS_COLONNES  → colonnes de la table (repérer une éventuelle historisation)")
print("REQ_GPS_UNICITE   → à quel niveau de clé l'unicité est atteinte")
print("REQ_GPS_CAUSES    → ce qui varie sur les clés en doublon")
print("REQ_GPS_EXEMPLES  → les 5 cas les plus chargés, en entier")
print("\nLecture de REQ_GPS_UNICITE : si `lignes` > `cles_soc_intercalaire`, la table")
print("de géocodage porte elle-même les doublons et la jointure ne fait que les")
print("propager. Si les deux sont égaux, le doublon vient de contrat_mgar.")

In [ ]:
PROJET_BQ = "matmut-dda-common-xpl-07c7"   # projet d'exécution / facturation
FICHIER_LOCAL = DOSSIER_INCENDIE / "export_societaires.csv"   # export manuel éventuel

COLONNES_MIN = ["id", "id_societaire", "numero_intercalaire",
                "lon_contrat_mgar", "lat_contrat_mgar", "code_sous_type"]


def charger_contrats() -> tuple[pd.DataFrame, str]:
    """BigQuery, sinon export local, sinon jeu simulé. Renvoie (df, mode)."""
    try:
        from google.cloud import bigquery
        df = bigquery.Client(project=PROJET_BQ).query(REQUETE_SQL).to_dataframe()
        return df, "bigquery"
    except Exception as exc:                                    # noqa: BLE001
        print(f"BigQuery indisponible ({type(exc).__name__}: {exc}).")

    if FICHIER_LOCAL.exists():
        df = pd.read_csv(FICHIER_LOCAL, dtype=str)
        for c in ("lon_contrat_mgar", "lat_contrat_mgar"):
            df[c] = pd.to_numeric(df[c], errors="coerce")
        return df, "fichier"

    print(f"Aucun export dans {FICHIER_LOCAL} → génération d'un jeu SIMULÉ.")
    return simuler_contrats(), "simulation"


def simuler_contrats(n_sur_bati: int = 180, n_perimetre: int = 260,
                     n_alentours: int = 900, graine: int = 20260727) -> pd.DataFrame:
    """Jeu de test synthétique : permet de dérouler tout le notebook sans données réelles."""
    rng = np.random.default_rng(graine)
    pts = []

    # a) contrats posés sur des bâtis relevés dans l'emprise brûlée
    ech = batis.sample(n_sur_bati, random_state=graine)
    for geom in ech.representative_point():
        d = rng.normal(0, 9, 2)
        pts.append((geom.x + d[0], geom.y + d[1]))

    # b) contrats dans le périmètre mais loin de tout bâti relevé
    for _ in range(n_perimetre):
        x0, y0, x1, y1 = contours.sample(1, random_state=int(rng.integers(1e6))).total_bounds
        pts.append((rng.uniform(x0, x1), rng.uniform(y0, y1)))

    # c) contrats des communes alentour, hors périmètre mais DANS l'emprise
    # interrogée : la requête pré-filtre sur cette bounding box, aucun contrat
    # réel ne peut se trouver au-delà.
    x0, y0, x1, y1 = contours.buffer(2_000).total_bounds
    for _ in range(n_alentours):
        pts.append((rng.uniform(x0, x1), rng.uniform(y0, y1)))

    g = gpd.GeoSeries([Point(x, y) for x, y in pts], crs=CRS_METRIQUE).to_crs(CRS_AFFICHAGE)
    n = len(g)
    sous_types = rng.choice(list("1234567") + ["8", "J", "A", "H"], n,
                            p=[.10, .12, .30, .07, .05, .18, .10, .03, .02, .02, .01])
    soc = [f"SIM{i:09d}" for i in range(n)]
    inter = rng.choice(["80", "81", "82"], n)
    return pd.DataFrame({
        "id_societaire": soc,
        "numero_intercalaire": inter,
        "id": [s + i for s, i in zip(soc, inter)],
        "numero_contrat": [f"C{i:06d}" for i in range(n)],
        "date_effet": pd.Timestamp("2024-01-01"),
        "rue_adresse_risque": "ADRESSE SIMULEE",
        "code_postal_adresse_risque": rng.choice(["33990", "40600", "33680"], n),
        "commune_adresse_risque": rng.choice(["LACANAU", "BISCARROSSE", "LE PORGE"], n),
        "lon_contrat_mgar": g.x.values,
        "lat_contrat_mgar": g.y.values,
        "code_sous_type": sous_types,
        "code_qualite_assure_habitation": rng.choice(["P", "L", "H"], n, p=[.60, .36, .04]),
        "code_type_bien": rng.choice(["MP", "IM", "MH"], n, p=[.62, .33, .05]),
        "level_contrat_mgar": rng.choice(
            ["housenumber", "street", "locality", "municipality"], n, p=[.74, .16, .04, .06]),
        "surface_habitable": rng.integers(35, 180, n).astype(str),
        "nombre_pieces_totales": rng.choice(["02", "03", "04", "05"], n),
        "montant_capital_mobilier": rng.choice([10_000, 15_000, 25_000, 35_000], n),
    })


contrats, MODE_SOURCE = charger_contrats()
contrats = contrats.drop(columns=["geom"], errors="ignore")   # WKT redondant avec lon/lat
manquantes = [c for c in COLONNES_MIN if c not in contrats.columns]
assert not manquantes, f"Colonnes absentes de l'export : {manquantes}"

# `id` = CONCAT(id_societaire, numero_intercalaire) : la jointure portant aussi sur le
# code postal, un même `id` peut théoriquement revenir sur deux adresses. On vérifie.
n_dup = int(contrats["id"].duplicated().sum())
print(f"\nMode = {MODE_SOURCE.upper()} — {len(contrats):,} lignes chargées".replace(",", " "))
print(f"{contrats['id'].nunique():,} identifiants contrat distincts".replace(",", " ")
      + (f" — ⚠️ {n_dup} ligne(s) en excès : voir le diagnostic ci-dessous." if n_dup else ""))
print("Colonnes facultatives présentes :",
      [c for c in ("code_type_bien", "commune_adresse_risque", "surface_habitable",
                   "nombre_pieces_totales", "montant_capital_mobilier")
       if c in contrats.columns] or "aucune")
contrats.head()

### Ce que mesure `REQ_GPS_UNICITE` — relevé du 27/07/2026

| Mesure | Valeur |
|---|---|
| Lignes dans l'emprise | **175 653** |
| Clés (sociétaire, intercalaire) | **16 818** |
| + code postal | 17 265 |
| + rue et commune | 18 046 |

**La table de géocodage porte 10,4 lignes par contrat.** L'adresse n'en explique
presque rien : ajouter code postal, rue et commune ne fait passer que de 16 818 à
18 046 clés, soit 1 228 contrats (7,3 %) à adresses multiples. **89,7 % du volume
reste dupliqué à adresse identique** — il existe donc une autre dimension dans cette
table : millésime IRIS, historisation technique, ou plusieurs sources de géocodage.
`REQ_GPS_COLONNES` et `REQ_GPS_EXEMPLES` la nomment.

**Cette duplication est sans effet sur nos résultats**, et c'est vérifiable : le
`SELECT DISTINCT` ramène 175 653 lignes à 16 086. Les colonnes que nous
sélectionnons — dont `lon_contrat_mgar` et `lat_contrat_mgar` — sont donc
constantes sur les ~9,7 lignes redondantes. La position géographique ne varie pas.

**Le vrai point d'attention est ailleurs.** L'emprise contient 16 818 contrats côté
géocodage, la jointure n'en restitue que 15 965 : **853 contrats, soit 5,1 %, ne
ressortent pas**. Deux causes possibles, aux conséquences opposées :

- le contrat n'est plus au stock courant de `contrat_mgar` — exclusion légitime ;
- son libellé d'adresse diffère entre les deux tables — **exclusion indue**, et donc
  des sociétaires potentiellement impactés absents du comptage.

`REQUETE_CONTROLE_JOINTURE` sépare les deux en comparant l'appariement à 3 clés et à
5 clés. À passer avant de communiquer les chiffres : 5 % pèse bien plus lourd que les
121 doublons d'identifiant (0,8 %).

### Ce que mesure `REQ_GPS_CAUSES` — même relevé

| Mesure | Valeur |
|---|---|
| Clés en doublon | **16 233** / 16 818, soit **96,5 % des contrats** |
| Lignes en excès | 158 835 — **90,4 % du volume de la table** |
| … strictement identiques | **15 179** (93,5 % des clés en doublon) |
| … dont la position varie | 954 |
| … adresse différente à position identique | **≥ 100** |

Trois enseignements, du moins grave au plus grave.

**93,5 % de la duplication est du bruit pur** : code postal, rue, commune et
coordonnées strictement identiques sur toutes les lignes d'un même contrat. La table
n'est pas au grain « contrat » — quelque chose d'autre varie (millésime IRIS,
historisation, source de géocodage). Sans effet sur l'analyse grâce au `DISTINCT`,
mais à signaler au producteur : 90 % du volume ne porte aucune information.

**954 contrats ont plusieurs positions distinctes.** C'est la vraie ambiguïté de
géocodage. Elle est traitée côté Python, en retenant la position la plus proche d'un
bâti de l'emprise.

**Au moins 100 contrats ont un point identique et un libellé d'adresse différent.**
C'est décisif : le texte d'adresse n'est pas stable entre les deux tables. Joindre
dessus écarte des contrats bien présents — d'où les 5,1 % perdus. C'est la raison
pour laquelle `JOINTURE_STRICTE_ADRESSE` vaut `False` par défaut : on privilégie le
recall au SQL, et on dédoublonne en Python avec la géométrie du feu, information que
la requête n'a pas.

### Comprendre les doublons d'identifiant

**La clé d'un contrat est le couple `(id_societaire, numero_intercalaire)`**, soit
notre champ `id`. Le numéro de contrat de `contrat_mgar` n'en est pas une : un
numéro distinct est attribué aux **dépendances**, si bien qu'un même contrat en
porte plusieurs. Il est donc écarté de l'analyse — le conserver multiplierait les
lignes sans rien ajouter.

La requête sort en `SELECT DISTINCT` : deux lignes qui partagent le même `id`
diffèrent donc forcément sur **au moins une colonne sélectionnée**. La cellule
suivante dit laquelle — c'est la seule façon de trancher, les causes possibles
n'ayant pas du tout les mêmes conséquences :

| Ce qui varie | Interprétation | Conséquence |
|---|---|---|
| `date_effet`, `numero_contrat` | contrats successifs sur la même clé | traité en amont : `QUALIFY` retient le plus récent |
| `level_contrat_mgar` et/ou la position | plusieurs géocodages concurrents du même contrat | on retient le plus précis |
| `lon_contrat_mgar` / `lat_contrat_mgar` seuls | même adresse géocodée deux fois | bénin — deux points au même endroit |
| `rue_` / `commune_` / `code_postal_adresse_risque` | plusieurs **lieux de risque** pour un même contrat, ou historique d'adresse resté dans la table gold | il faut choisir la bonne position |
| `code_sous_type`, `code_qualite_assure_habitation`… | deux lignes courantes dans `contrat_mgar` malgré `tech_date_fin_historisation IS NULL` | anomalie d'historisation à remonter au producteur |

Quelle que soit la cause, on ne peut pas laisser un contrat produire plusieurs points :
il compterait plusieurs fois sur la carte et dans l'export. Le traitement retenu plus
bas conserve, pour chaque `id`, le **géocodage le plus précis** au sens de
`level_contrat_mgar` : `housenumber` d'abord, puis `street`, `locality`,
`municipality`.

Mais **le niveau ne suffit pas** : sur les données réelles, il ne départage qu'une
clé sur trois. Les autres ont plusieurs positions **au même niveau** — jusqu'à quatre
`housenumber` pour une seule adresse, distantes de plus de 3 km. Un second critère
est donc appliqué : parmi les candidats du meilleur niveau, on retient la **position
la plus centrale**, c'est-à-dire le consensus, ce qui écarte les valeurs aberrantes.

Ces deux critères ont en commun d'être **neutres**. Départager sur la distance au
bâti — ce que faisait une première version — revenait à retenir systématiquement la
position la plus incriminante, donc à surestimer l'impact par construction.

Reste que lorsque les candidats sont distants de kilomètres, aucun critère ne produit
une localisation fiable : le choix est par défaut. L'écart entre positions candidates
est donc mesuré, et tout contrat au-delà de `SEUIL_ECART_POSITIONS_M` (100 m) est
marqué `position_incertaine` dans l'export, avec son écart en mètres.

Reste une réserve : à **adresses différentes**, choisir un niveau revient à choisir un
lieu de risque, ce qu'aucune règle automatique ne peut fonder. Ces contrats sont
comptés à part et marqués `id_plusieurs_adresses` dans l'export.

In [ ]:
# La clé d'un contrat est la double clé (id_societaire, numero_intercalaire).
# `id` n'en est que la concaténation, pratique pour grouper mais illisible dans
# un tableau : les deux composantes sont affichées séparément.
CLE_CONTRAT = ["id_societaire", "numero_intercalaire"]
# L'adresse est toujours affichée, même lorsqu'elle ne varie pas : depuis que le
# contrat retenu la fournit, elle est identique sur toutes les lignes d'une clé et
# disparaîtrait donc des « colonnes qui diffèrent ». Or c'est elle qui rend le
# relevé exploitable — sans elle on voit qu'un géocodage cloche, pas sur quoi.
COLONNES_ADRESSE = ["rue_adresse_risque", "code_postal_adresse_risque",
                    "commune_adresse_risque"]

doublons_id = contrats[contrats["id"].duplicated(keep=False)]

if doublons_id.empty:
    print("Aucun doublon : la clé (id_societaire, numero_intercalaire) est unique ici.")
else:
    n_id = doublons_id["id"].nunique()
    print(f"{len(doublons_id)} lignes portent {n_id} clés (id_societaire, "
          f"numero_intercalaire) dupliquées "
          f"({len(contrats) - contrats['id'].nunique()} lignes en excès).\n")

    # Pour chaque colonne : sur combien d'identifiants dupliqués prend-elle
    # plusieurs valeurs ? Les colonnes de tête sont la cause du doublon.
    varie = (doublons_id.groupby("id").nunique(dropna=False) > 1).sum()
    varie = varie[varie > 0].sort_values(ascending=False)
    display(pd.DataFrame({
        "identifiants concernés": varie,
        "part": (varie / n_id).map("{:.0%}".format),
    }))

    # Deux cas concrets, réduits aux colonnes qui varient effectivement : le plus
    # chargé, et le plus chargé parmi ceux à adresses multiples s'il diffère.
    _adresse_dispo = [c for c in COLONNES_ADRESSE if c in doublons_id.columns]

    def _montrer(ex_id, titre):
        ex = doublons_id[doublons_id["id"] == ex_id]
        cols_var = [c for c in ex.columns
                    if c not in CLE_CONTRAT + _adresse_dispo
                    and ex[c].nunique(dropna=False) > 1]
        adresse_varie = [c for c in _adresse_dispo if ex[c].nunique(dropna=False) > 1]
        print(f"\n{titre} — sociétaire {ex[CLE_CONTRAT[0]].iloc[0]}, intercalaire "
              f"{ex[CLE_CONTRAT[1]].iloc[0]} ({len(ex)} lignes)")
        print(f"  colonnes qui diffèrent : "
              f"{(adresse_varie + cols_var) or 'aucune (lignes identiques)'}")
        if _adresse_dispo and not adresse_varie:
            print("  adresse identique sur toutes les lignes : le doublon vient du "
                  "géocodage, pas du contrat.")
        display(ex[CLE_CONTRAT + _adresse_dispo + cols_var])
        # Deux lignes identiques sur toutes les colonnes malgré le SELECT DISTINCT
        # signaleraient un doublon dans la source, pas un écart de contenu.
        if not cols_var and not adresse_varie:
            print("  ⚠️  Lignes strictement identiques : doublon de la source, "
                  "à remonter au producteur.")

    _montrer(doublons_id["id"].value_counts().index[0], "Cas le plus chargé")

    _adr = ["rue_adresse_risque", "code_postal_adresse_risque", "commune_adresse_risque"]
    _adr = [c for c in _adr if c in doublons_id.columns]
    if _adr:
        n_adr = doublons_id.groupby("id")[_adr].nunique().max(axis=1)
        multi = n_adr[n_adr > 1]
        if not multi.empty and multi.index[0] != doublons_id["id"].value_counts().index[0]:
            _montrer(multi.sort_values(ascending=False).index[0],
                     "Cas à adresses de risque multiples")
        if not multi.empty:
            print(f"\n{len(multi)} clé(s) portent plusieurs adresses de risque : "
                  f"plusieurs lieux assurés, ou historique d'adresse resté dans la "
                  f"table gold. Le choix de position y est un arbitrage par défaut, "
                  f"marqué `id_plusieurs_adresses` dans l'export.")

## 4. Segmentation RP / RS / PNO

Le type de contrat se lit sur **`code_sous_type`** (source
`cd_opt_mgar_of_donnees_sas_mgar_of_cwhymgaz`), regroupement fonctionnel documenté
dans le dictionnaire `CONTRAT_MGAR` :

| `code_sous_type` | Segment |
|---|---|
| `1` `2` `3` `4` `5` | **RP** — résidence principale |
| `6` | **PNO** — propriétaire non occupant |
| `7` | **RS** — résidence secondaire |
| `8` `9` `J` | JEUN — contrat jeune |
| `A` `B` `C` `D` `P` `R` | ETUD — étudiant |
| `E` `F` | ETUE — étudiant étranger |
| `H` | HEB — hébergé |
| `T` | TBNH — temporairement bien non habité |

La demande porte sur **RP / RS / PNO**. Les autres segments sont malgré tout comptés
et affichés à part : ce sont aussi des logements occupés, et les exclure sans le dire
minorerait le nombre de sociétaires touchés.

In [ ]:
MAP_SOUS_TYPE = {
    **{c: "RP" for c in "12345"},
    "6": "PNO",
    "7": "RS",
    **{c: "JEUN" for c in "89J"},
    **{c: "ETUD" for c in "ABCDPR"},
    **{c: "ETUE" for c in "EF"},
    "H": "HEB",
    "T": "TBNH",
}
LIB_SEGMENT = {
    "RP": "Résidence principale", "RS": "Résidence secondaire",
    "PNO": "Propriétaire non occupant", "JEUN": "Contrat jeune",
    "ETUD": "Étudiant", "ETUE": "Étudiant étranger",
    "HEB": "Hébergé", "TBNH": "Bien temporairement non habité",
    "INCONNU": "Sous-type absent ou non référencé",
}
# Le dictionnaire documente les modalités sous la forme {"IM": "'APPARTEMENT'"} sans
# préciser si la colonne stocke le code ou le libellé. Les deux sont donc acceptés :
# se tromper ferait basculer toute la population en « non renseigné ».
LIB_TYPE_BIEN = {
    "IM": "Immeuble",       "IMMEUBLE": "Immeuble", "APPARTEMENT": "Immeuble",
    "MP": "Maison particulière", "MAISON PART.": "Maison particulière",
    "MAISON PART": "Maison particulière", "MAISON PARTICULIERE": "Maison particulière",
    "MH": "Mobile home",    "MOBILE HOME": "Mobile home",
}
# Regroupement des modalités de `code_qualite_assure_habitation` en trois postes.
# Le critère est « qui supporte le dommage au bâti » : nu-propriétaire et usufruitier
# sont du côté propriétaire, les colocations du côté locataire. Hébergé gratuit,
# logement de service et chambres en établissement ne relèvent ni de l'un ni de
# l'autre — les classer arbitrairement fausserait la lecture, ils restent à part.
# Codes et libellés acceptés, pour la même raison que LIB_TYPE_BIEN.
STATUT_OCCUPATION = {
    "P": "Propriétaire", "PROPRIETAIRE": "Propriétaire",
    "N": "Propriétaire", "NU-PROPRIET.": "Propriétaire", "NU-PROPRIET": "Propriétaire",
    "U": "Propriétaire", "USUFRUITIER": "Propriétaire",
    "L": "Locataire", "LOCATAIRE": "Locataire",
    "I": "Locataire", "COL CT INDIV": "Locataire",
    "G": "Locataire", "COL CT COM": "Locataire", "COLOC": "Locataire",
    "H": "Autre / non renseigné", "HEB GRATUIT": "Autre / non renseigné",
    "C": "Autre / non renseigné", "LOGT SERVICE": "Autre / non renseigné",
    "R": "Autre / non renseigné", "CH MAIS RET": "Autre / non renseigné",
    "M": "Autre / non renseigné", "CH MEDICAL.": "Autre / non renseigné",
    "CH MEDICAL": "Autre / non renseigné",
    "S": "Autre / non renseigné", "SANS RESID": "Autre / non renseigné",
    "CH INST SPEC": "Autre / non renseigné",
}
ORDRE_STATUTS = ["Propriétaire", "Locataire", "Autre / non renseigné"]

LIB_QUALITE = {"P": "Propriétaire", "L": "Locataire", "H": "Hébergé gratuit",
               "I": "Colocation individuelle", "R": "Chambre maison de retraite",
               "M": "Chambre établissement médical", "G": "Colocation commune",
               "N": "Nu-propriétaire", "C": "Logement de service",
               "U": "Usufruitier", "S": "Sans résidence fixe"}
LIB_QUALITE.update({  # libellés, au cas où la colonne les stocke en clair
    "PROPRIETAIRE": "Propriétaire", "LOCATAIRE": "Locataire",
    "HEB GRATUIT": "Hébergé gratuit", "COL CT INDIV": "Colocation individuelle",
    "CH MAIS RET": "Chambre maison de retraite", "CH MEDICAL.": "Chambre établissement médical",
    "COL CT COM": "Colocation commune", "NU-PROPRIET.": "Nu-propriétaire",
    "LOGT SERVICE": "Logement de service", "USUFRUITIER": "Usufruitier",
    "SANS RESID": "Sans résidence fixe", "CH INST SPEC": "Chambre établissement spécialisé",
})

contrats["code_sous_type"] = contrats["code_sous_type"].astype("string").str.strip().str.upper()
contrats["segment"] = contrats["code_sous_type"].map(MAP_SOUS_TYPE).fillna("INCONNU")
contrats["dans_perimetre_demande"] = contrats["segment"].isin(SEGMENTS_CIBLE)
def _mapper(colonne: str, table: dict, cible: str) -> None:
    """Applique un mapping et signale toute modalité inconnue plutôt que de la
    ranger en « non renseigné » : une modalité muette fausserait tous les
    comptages sans laisser de trace."""
    if colonne not in contrats:
        return
    brut = contrats[colonne].astype("string").str.strip().str.upper()
    contrats[cible] = brut.map(table).fillna("Non renseigné")
    inconnues = sorted(set(brut.dropna().unique()) - set(table) - {""})
    if inconnues:
        n_lignes = int(brut.isin(inconnues).sum())
        print(f"⚠️  {colonne} : modalité(s) non reconnue(s) {inconnues} "
              f"({n_lignes} lignes, {n_lignes / len(contrats):.1%}) → classées "
              f"« Non renseigné ». Compléter la table de correspondance.")


_mapper("code_type_bien", LIB_TYPE_BIEN, "type_bien")
_mapper("code_qualite_assure_habitation", LIB_QUALITE, "qualite")
if "code_qualite_assure_habitation" in contrats:
    q = contrats["code_qualite_assure_habitation"].astype("string").str.strip().str.upper()
    # Regroupement en 3 postes pour la carte et les compteurs. Le détail des 9+
    # modalités reste disponible dans `qualite` et dans l'export CSV.
    contrats["statut"] = q.map(STATUT_OCCUPATION).fillna("Autre / non renseigné")

repartition = (contrats["segment"].value_counts(dropna=False).rename("contrats")
               .to_frame()
               .assign(part=lambda d: (d.contrats / d.contrats.sum()).map("{:.1%}".format),
                       libelle=lambda d: d.index.map(LIB_SEGMENT),
                       demande=lambda d: np.where(d.index.isin(SEGMENTS_CIBLE), "✅", "—")))
display(repartition[["libelle", "contrats", "part", "demande"]])

## 5. Sinistres incendie déclarés

Le rapprochement géométrique dit qui *pourrait* être touché. Un **sinistre incendie
déclaré** dit qui l'est. Croiser les deux transforme le livrable : on ne produit plus
seulement une estimation, on produit une **matrice de validation** — et surtout une
liste de sociétaires géométriquement impactés **qui n'ont pas encore déclaré**, à
contacter en priorité.

Table : `matmut-dni-datalake-prd-8ec7.silver_sinistreetprestation_sigmainframe_prd.situation_sinistre_mgar`.

**L'ordre des filtres compte.** `est_clos` évolue d'un mouvement à l'autre : filtrer
sur « dossier en cours » avant d'avoir réduit chaque sinistre à son dernier mouvement
ferait ressortir des dossiers ouverts au mouvement 2 mais clos au mouvement 7. La
réduction se fait donc d'abord, dans une CTE, et les filtres métier ensuite.

Jointure sur `(id_reference_societaire, numero_intercalaire_contrat)`, soit la même
clé métier que partout ailleurs. Trois filtres :

- `tech_date_fin_historisation IS NULL` — la table est historisée ; sans ce filtre on
  ramènerait aussi les versions périmées ;
- **dernier `numero_mouvement` par sinistre** — même en écartant l'historique, un
  dossier conserve tous ses mouvements, du premier au dernier. C'est le dernier qui
  donne la vision à jour ;
- `date_enregistrement >= 2026-07-23`, départ du feu de Gironde — celui de
  Biscarrosse ayant démarré le 24. Critère **stable dans le temps**. On retient la
  date d'enregistrement plutôt que la date de survenance : la première est posée par
  le système à l'ouverture du dossier, la seconde est déclarative, donc sujette aux
  saisies approximatives ou par défaut. Un sinistre ne pouvant être enregistré avant
  d'être survenu, le seuil reste bien couvrant. `date_survenance` est malgré tout
  ramenée dans le résultat, pour contrôle ;
- `code_descriptif_sinistre = '05'`, l'incendie ;
- `est_clos = FALSE` — dossier en cours (`0` = en cours, `1` = clos).

La fenêtre glissante « ouvert dans les 15 derniers jours » n'est plus appliquée :
elle servait de proxy à « depuis le feu », rôle que la date de survenance remplit
mieux et sans dériver. La conserver reviendrait, dans quelques semaines, à écarter
des déclarations tardives portant pourtant sur ces mêmes feux.
`FENETRE_SINISTRE_JOURS = 15` la rétablit si besoin.

Un dossier déjà clos reste une preuve d'impact : `SINISTRES_OUVERTS_UNIQUEMENT = False`
les réintègre, au cas où la question passerait de « qui gère-t-on » à « qui a été
touché ».

Le descriptif **incendie est `code_descriptif_sinistre = '05'`** — non documenté dans
le dictionnaire, qui se contente d'exemples (`01`, `02`, `06`, `23`, `42`), et confirmé
par le métier. `REQ_CODES_SINISTRE` reste disponible pour le vérifier sur pièce : sur
les sinistres survenus depuis le 20/07/2026 en Gironde et dans les Landes, `05` doit
dominer nettement.

`numero_evenement_grande_ampleur` mérite un regard : si ces feux ont reçu un numéro
d'événement, filtrer dessus serait **plus fiable** que par code descriptif — c'est
exactement ce à quoi ce champ sert.

In [ ]:
# La table sinistres vit dans son propre dataset, distinct de celui de contrat_mgar.
TABLE_SINISTRES = ("matmut-dni-datalake-prd-8ec7."
                   "silver_sinistreetprestation_sigmainframe_prd.situation_sinistre_mgar")

# Critère retenu : sinistres incendie survenus depuis le départ du premier feu.
# La fenêtre glissante sur la date d'ouverture n'était qu'un proxy de « depuis le
# feu » ; la date de survenance est plus juste et ne dérive pas. La laisser
# activée écarterait les déclarations tardives, qui concernent pourtant bien ces
# feux. Mettre un entier pour la réactiver.
FENETRE_SINISTRE_JOURS = None

# `est_clos = FALSE` répond à la demande « sinistre ouvert ». Passer à False pour
# compter aussi les dossiers déjà clos : ils restent une preuve d'impact, même si
# la gestion est terminée.
SINISTRES_OUVERTS_UNIQUEMENT = True
# Code incendie de `code_descriptif_sinistre`, confirmé par le métier.
# Mettre None pour interroger tous les descriptifs — le rapprochement resterait
# exploitable, mais ne serait plus spécifique à l'incendie.
CODES_SINISTRE_INCENDIE = ("05",)
FICHIER_LOCAL_SINISTRES = DOSSIER_INCENDIE / "export_sinistres.csv"

# Identifier le code « incendie » : après le 20/07/2026 en Gironde et dans les
# Landes, le descriptif dominant ne peut être que celui-là.
REQ_CODES_SINISTRE = f"""
SELECT
  code_descriptif_sinistre,
  COUNT(*)                        AS sinistres,
  MIN(date_survenance)            AS premiere_survenance,
  MAX(date_survenance)            AS derniere_survenance,
  COUNT(DISTINCT numero_evenement_grande_ampleur) AS evenements_grande_ampleur
FROM `{TABLE_SINISTRES}`
WHERE tech_date_fin_historisation IS NULL
  AND date_survenance >= DATE '2026-07-20'
  AND code_departement_survenance IN ('33', '40')
GROUP BY code_descriptif_sinistre
ORDER BY sinistres DESC
"""

_ouverts = "\n  AND s.est_clos = FALSE" if SINISTRES_OUVERTS_UNIQUEMENT else ""
_fenetre = ("" if not FENETRE_SINISTRE_JOURS else
            f"\n  AND s.date_enregistrement >= "
            f"DATE_SUB(CURRENT_DATE(), INTERVAL {FENETRE_SINISTRE_JOURS} DAY)")
_filtre_codes = ("" if not CODES_SINISTRE_INCENDIE else
                 "\n  AND s.code_descriptif_sinistre IN ("
                 + ", ".join(f"'{c}'" for c in CODES_SINISTRE_INCENDIE) + ")")

# L'ordre des filtres n'est pas indifférent. `tech_date_fin_historisation IS NULL`
# ne suffit pas : un sinistre garde plusieurs mouvements courants, du 1er au dernier.
# Il faut d'abord réduire chaque dossier à son dernier `numero_mouvement`, puis
# seulement filtrer — `est_clos` change d'un mouvement à l'autre, et filtrer avant
# ferait ressortir un dossier ouvert au mouvement 2 mais clos au mouvement 7.
REQUETE_SINISTRES = f"""
WITH derniere_situation AS (
  SELECT *
  FROM `{TABLE_SINISTRES}`
  WHERE tech_date_fin_historisation IS NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY numero_sinistre, cle_controle_numero_sinistre
    -- `numero_mouvement` est cadré à gauche par des zéros ('0000', '0004') : le tri
    -- texte suffirait, le cast protège d'une valeur non cadrée.
    ORDER BY SAFE_CAST(numero_mouvement AS INT64) DESC, numero_mouvement DESC
  ) = 1
)
SELECT DISTINCT
  s.id_reference_societaire       AS id_societaire,
  s.numero_intercalaire_contrat   AS numero_intercalaire,
  s.numero_sinistre,
  s.numero_mouvement,
  s.code_descriptif_sinistre,
  s.date_enregistrement,
  s.date_survenance,
  s.numero_evenement_grande_ampleur
FROM derniere_situation s
-- Enregistrement depuis le départ du premier feu. `date_enregistrement` est posée
-- par le système à l'ouverture du dossier, là où `date_survenance` est déclarative
-- donc moins fiable. Un sinistre ne pouvant être enregistré avant d'être survenu,
-- ce seuil capte bien les déclarations liées aux feux.
WHERE s.date_enregistrement >= DATE '{DATE_DEBUT_FEUX}'{_ouverts}{_fenetre}{_filtre_codes}
"""
print(REQUETE_SINISTRES)
if not CODES_SINISTRE_INCENDIE:
    print("⚠️  CODES_SINISTRE_INCENDIE non renseigné : tous les descriptifs sont retenus.")
    print("    Passer REQ_CODES_SINISTRE dans BigQuery pour identifier le code incendie.")

In [ ]:
def simuler_sinistres(part_declarants: float = 0.62, bruit: float = 0.004,
                      graine: int = 424242) -> pd.DataFrame:
    """Sinistres simulés, corrélés à la proximité d'un bâti de l'emprise brûlée —
    sinon la matrice de validation n'aurait aucun sens."""
    rng = np.random.default_rng(graine)
    g = gpd.GeoDataFrame(
        contrats[["id_societaire", "numero_intercalaire"]],
        geometry=gpd.points_from_xy(contrats["lon_contrat_mgar"], contrats["lat_contrat_mgar"]),
        crs=CRS_AFFICHAGE).to_crs(CRS_METRIQUE)
    proche = g.geometry.apply(lambda pt: batis.distance(pt).min() <= 30.0)
    tire = np.where(proche, rng.random(len(g)) < part_declarants, rng.random(len(g)) < bruit)
    sel = contrats.loc[tire, ["id_societaire", "numero_intercalaire"]].copy()
    sel["numero_sinistre"] = [f"26SIM{i:04d}" for i in range(len(sel))]
    sel["numero_mouvement"] = "0002"
    sel["code_descriptif_sinistre"] = CODES_SINISTRE_INCENDIE[0] if CODES_SINISTRE_INCENDIE else "05"
    sel["date_enregistrement"] = pd.Timestamp(DATE_DEBUT_FEUX) + pd.Timedelta(days=2)
    sel["date_survenance"] = pd.Timestamp(DATE_DEBUT_FEUX)
    sel["numero_evenement_grande_ampleur"] = "2026018"
    return sel


def charger_sinistres() -> tuple[pd.DataFrame, str]:
    if MODE_SOURCE == "bigquery":
        try:
            from google.cloud import bigquery
            return bigquery.Client(project=PROJET_BQ).query(REQUETE_SINISTRES).to_dataframe(), "bigquery"
        except Exception as exc:                                 # noqa: BLE001
            print(f"Sinistres indisponibles dans BigQuery ({type(exc).__name__}: {exc}).")
    if FICHIER_LOCAL_SINISTRES.exists():
        return pd.read_csv(FICHIER_LOCAL_SINISTRES, dtype=str), "fichier"
    print("Aucun export de sinistres → jeu SIMULÉ.")
    return simuler_sinistres(), "simulation"


sinistres, MODE_SINISTRES = charger_sinistres()

# Un contrat peut porter plusieurs sinistres : on ramène à un indicateur par contrat.
for _c in ("id_societaire", "numero_intercalaire"):
    sinistres[_c] = sinistres[_c].astype("string").str.strip()

agg = (sinistres.groupby(["id_societaire", "numero_intercalaire"], dropna=False)
                .agg(nb_sinistres_ouverts=("numero_sinistre", "nunique"),
                     numeros_sinistres=("numero_sinistre",
                                        lambda x: ", ".join(sorted(set(x.dropna()))[:3])),
                     date_declaration=("date_enregistrement", "min"))
                .reset_index())

_avant_cols = set(contrats.columns)
contrats = contrats.merge(agg, how="left", on=["id_societaire", "numero_intercalaire"])
contrats["nb_sinistres_ouverts"] = contrats["nb_sinistres_ouverts"].fillna(0).astype(int)
contrats["sinistre_declare"] = contrats["nb_sinistres_ouverts"] > 0

print(f"\nMode sinistres = {MODE_SINISTRES.upper()} — {len(sinistres)} sinistre(s) ouvert(s) "
      f"sur {len(agg)} contrat(s) distinct(s).")
print(f"{contrats['sinistre_declare'].sum()} contrat(s) de l'emprise interrogée portent un "
      f"sinistre ouvert ({contrats['sinistre_declare'].mean():.1%}).")
_orphelins = len(agg) - contrats["sinistre_declare"].sum()
if _orphelins > 0:
    print(f"⚠️  {_orphelins} contrat(s) sinistrés ne sont pas dans l'extraction contrats : "
          f"hors emprise géographique, ou absents du stock courant de contrat_mgar.")

## 6. Qualité du géocodage

Étape indispensable avant de compter : si une partie des contrats est géocodée au
**centroïde de la commune** plutôt qu'à l'adresse, tout point tombant par hasard près
d'un bâti de l'emprise produirait un faux positif.

La table de géocodage répond directement à la question, via la colonne
**`level_contrat_mgar`**, qui reprend les niveaux de la Base Adresse Nationale :

| Modalité | Où le point est posé | Écart typique au bâti | Exploitable ? |
|---|---|---|---|
| `housenumber` | sur le point adresse | quelques mètres | **oui** |
| `street` | sur l'axe de la voie | 10 à 50 m | oui, avec réserve |
| `locality` | au centre d'un lieu-dit | centaines de mètres | **non** |
| `municipality` | au centroïde de la commune | kilomètres | **non** |

C'est cette colonne qui fait foi. L'heuristique ci-dessous lui est confrontée dans un
tableau de concordance, mais ne sert plus qu'à documenter l'écart.

Elle reste utile pour une autre raison : partager une coordonnée n'est pas un défaut
en soi — **un immeuble de 30 lots produit légitimement 30 contrats au même point**.
Le discriminant est le **nombre de voies distinctes** sur une même position :

- une seule voie → immeuble plausible, position fiable ;
- plusieurs voies sur le même point → géométriquement impossible, le géocodage est
  retombé sur un centroïde de commune ou de voie.

La part d'appartements dans chaque groupe sert de contrôle croisé : elle doit être
nettement plus élevée du côté « immeuble ».

In [ ]:
contrats = contrats.dropna(subset=["lon_contrat_mgar", "lat_contrat_mgar"]).copy()

pts = gpd.GeoDataFrame(
    contrats,
    geometry=gpd.points_from_xy(contrats["lon_contrat_mgar"], contrats["lat_contrat_mgar"]),
    crs=CRS_AFFICHAGE,
).to_crs(CRS_METRIQUE)

SEUIL_COORD_PARTAGEE = 5      # nb de contrats à partir duquel on regarde de près

pts["cle_xy"] = (pts.geometry.x.round(0).astype(int).astype(str) + "_"
                 + pts.geometry.y.round(0).astype(int).astype(str))
pts["rue_norm"] = (pts.get("rue_adresse_risque", pd.Series("", index=pts.index))
                   .astype("string").str.upper().str.replace(r"\s+", " ", regex=True).str.strip())

g = pts.groupby("cle_xy")
nb_contrats_xy = g["id"].transform("size")
nb_rues_xy = g["rue_norm"].transform("nunique")

# Partager une coordonnée n'est pas un défaut en soi : un immeuble de 30 lots
# produit légitimement 30 contrats au même point. Ce qui trahit un géocodage
# retombé sur un centroïde de commune ou de voie, c'est que des adresses de
# RUES DIFFÉRENTES se retrouvent sur le même point — géométriquement impossible.
pts["coord_partagee"] = nb_contrats_xy >= SEUIL_COORD_PARTAGEE
pts["geocodage_suspect"] = pts["coord_partagee"] & (nb_rues_xy > 1)
pts["coord_immeuble"] = pts["coord_partagee"] & (nb_rues_xy == 1)

print(f"{len(pts):,} contrats géolocalisés".replace(",", " "))
print(f"{pts['cle_xy'].nunique():,} positions distinctes\n".replace(",", " "))

recap = pd.DataFrame([
    {"cas": f"Coordonnée portant ≥ {SEUIL_COORD_PARTAGEE} contrats, une seule voie",
     "positions": int(pts.loc[pts["coord_immeuble"], "cle_xy"].nunique()),
     "contrats": int(pts["coord_immeuble"].sum()),
     "lecture": "immeuble plausible — normal"},
    {"cas": f"Coordonnée portant ≥ {SEUIL_COORD_PARTAGEE} contrats, plusieurs voies",
     "positions": int(pts.loc[pts["geocodage_suspect"], "cle_xy"].nunique()),
     "contrats": int(pts["geocodage_suspect"].sum()),
     "lecture": "centroïde commune/voie — position non fiable"},
])
recap["part des contrats"] = (recap["contrats"] / len(pts)).map("{:.1%}".format)
display(recap.set_index("cas"))

if "type_bien" in pts:
    print("Contrôle croisé — part de biens en immeuble selon le cas :")
    for lbl, masque in [("immeuble plausible", pts["coord_immeuble"]),
                        ("centroïde probable", pts["geocodage_suspect"]),
                        ("position isolée", ~pts["coord_partagee"])]:
        sous = pts.loc[masque, "type_bien"]
        if sous.empty:
            print(f"  {lbl:<22}      — aucun contrat dans ce cas")  # noqa: E501
            continue
        part = (sous == "Immeuble").mean()
        print(f"  {lbl:<22} {part:6.1%} en immeuble ({len(sous):,} contrats)".replace(",", " "))
    print("  → une part élevée de biens en immeuble conforte l'hypothèse « immeuble ».")

# --------------------------------------------------------------------------- #
# Si la table de géocodage expose la précision du point, elle fait foi : on
# remplace l'heuristique par la donnée, en gardant la comparaison des deux.
# --------------------------------------------------------------------------- #
COL_PRECISION = next((c for c in CANDIDATS_COLONNE_PRECISION if c in pts.columns), None)

if COL_PRECISION is None:
    print("\nAucune colonne de précision de géocodage dans l'export.")
    print("→ passer REQ_GPS_COLONNE_PRECISION dans BigQuery pour la localiser, puis")
    print("  l'ajouter à COLONNES_OPTIONNELLES : elle fait foi sur l'heuristique.")
else:
    print(f"\nColonne de précision détectée : `{COL_PRECISION}` — elle fait foi.")
    modalite = pts[COL_PRECISION].astype("string").str.strip().str.lower()
    pts["precision_geocodage"] = modalite

    def _classer(v):
        if pd.isna(v) or v == "":
            return "Non renseigné"
        if v in MODALITES_CENTROIDE_COMMUNE:
            return "Centroïde commune"
        if v in MODALITES_LIEU_DIT:
            return "Lieu-dit"
        if v in MODALITES_VOIE:
            return "Voie"
        if v in MODALITES_ADRESSE:
            return "Adresse exacte"
        return f"Non classé : {v}"

    pts["niveau_geocodage"] = modalite.map(_classer)
    display(pts["niveau_geocodage"].value_counts(dropna=False)
              .rename("contrats").to_frame()
              .assign(part=lambda d: (d.contrats / len(pts)).map("{:.1%}".format)))

    non_classes = [m for m in modalite.dropna().unique() if _classer(m).startswith("Non classé")]
    if non_classes:
        print(f"⚠️  Modalités non reconnues : {non_classes}")
        print("   → les ajouter au bon ensemble MODALITES_* ci-dessus avant de conclure.")

    # La donnée remplace l'heuristique. On conserve celle-ci pour comparaison.
    pts["geocodage_suspect_heuristique"] = pts["geocodage_suspect"]
    pts["geocodage_suspect"] = pts["niveau_geocodage"].isin(["Centroïde commune", "Lieu-dit"])
    print(f"\nPositions non fiables : {pts['geocodage_suspect'].sum():,} contrats "
          f"({pts['geocodage_suspect'].mean():.1%})".replace(",", " "))
    print("Concordance avec l'heuristique multi-voies :")
    display(pd.crosstab(pts["geocodage_suspect_heuristique"], pts["geocodage_suspect"],
                        rownames=["heuristique"], colnames=["colonne source"]))

if pts["geocodage_suspect"].any():
    pire = pts.loc[pts["geocodage_suspect"]].groupby("cle_xy")["rue_norm"].agg(["size", "nunique"])
    pire = pire.sort_values("size", ascending=False).head(3)
    print("\nPositions les plus chargées parmi les centroïdes probables :")
    for cle, r in pire.iterrows():
        rues = pts.loc[pts["cle_xy"] == cle, "rue_norm"].unique()[:3]
        print(f"  {int(r['size'])} contrats, {int(r['nunique'])} voies — ex. : {', '.join(rues)}")

## 7. Appariement spatial

Pour chaque contrat, on cherche le **bâtiment le plus proche relevé dans l'emprise brûlée** puis on classe :

| Niveau | Règle | Lecture |
|---|---|---|
| **Certain — dans l'emprise brûlée** | le point tombe **dans** l'emprise d'un bâti brûlé | le logement assuré est l'un des bâtiments de l'emprise |
| **Très probable — à moins de 10 m** | ≤ 10 m d'un bâti brûlé | décalage de géocodage courant (bord de parcelle) |
| **Probable — à moins de 25 m** | ≤ 25 m d'un bâti brûlé | tolérance haute du géocodage adresse |
| **Exposé — dans le périmètre du feu** | dans le contour, > 25 m de tout bâti brûlé | exposé, bâti non relevé dans l'emprise |
| **Hors périmètre** | reste | non concerné |

Les trois premiers niveaux constituent la réponse à « **effectivement impactés** ».
Le tableau de sensibilité plus bas montre l'effet du seuil sur le compte.

### Le géocodage borne ce qu'une distance autorise à conclure

Une distance ne vaut que ce que vaut la position dont elle part. `level_contrat_mgar`
sert donc de garde-fou, avant même le calcul :

- **`municipality` et `locality`** — le point est au centroïde de la commune ou d'un
  lieu-dit. À cette échelle, sa distance à un bâtiment est du pur hasard. Ces contrats
  sont **sortis du comptage** et suivis à part, sous « position trop imprécise pour
  conclure ». Les compter reviendrait à fabriquer des faux positifs : le centroïde
  d'une commune sinistrée tombe mécaniquement près des bâtis brûlés.
- **`street`** — le point est sur l'axe de la voie, pas sur le bâti. Tomber dans une
  emprise y est une coïncidence géométrique, pas une preuve : ces contrats sont
  **plafonnés à « très probable »**, jamais « certain ».
- **`housenumber`** — seul niveau où les seuils s'appliquent tels quels.

Ces deux règles se désactivent par `PLAFONNER_NIVEAU_VOIE` et
`NIVEAUX_GEOCODAGE_INEXPLOITABLES`. Elles rendent le compte plus prudent, donc plus
petit : c'est voulu — un chiffre communiqué doit pouvoir être défendu contrat par
contrat.

In [ ]:
BAT_COLS = ["bat_id", "feu", "surface_bati_m2", "geometry"]

# Bâti relevé le plus proche, dans la limite du seuil le plus large
appar = gpd.sjoin_nearest(
    pts, batis[BAT_COLS], how="left",
    max_distance=max(SEUILS_SENSIBILITE_M), distance_col="distance_bati_m",
)
# sjoin_nearest peut renvoyer plusieurs ex-aequo : on garde le plus proche
appar = (appar.sort_values("distance_bati_m")
              .loc[~appar.index.duplicated(keep="first")]
              .sort_index()
              .drop(columns=["index_right"], errors="ignore"))

# Appartenance au périmètre du feu
peri = gpd.sjoin(pts[["geometry"]], contours[["feu", "geometry"]],
                 how="left", predicate="within")
peri = peri.loc[~peri.index.duplicated(keep="first")]
appar["feu_perimetre"] = peri["feu"]


def niveau(r):
    d = r["distance_bati_m"]
    if pd.notna(d):
        if d <= 0:
            return "Certain — dans l'emprise brûlée"
        if d <= SEUIL_TRES_PROBABLE_M:
            return "Très probable — à moins de 10 m"
        if d <= SEUIL_PROBABLE_M:
            return "Probable — à moins de 25 m"
    if pd.notna(r["feu_perimetre"]):
        return "Exposé — dans le périmètre du feu"
    return "Hors périmètre"


appar["niveau_impact"] = appar.apply(niveau, axis=1)
appar["feu_rattache"] = appar["feu"].fillna(appar["feu_perimetre"])
# Le niveau de géocodage borne ce qu'on a le droit de conclure d'une distance.
if COL_PRECISION:
    _geo = appar["niveau_geocodage"]

    if PLAFONNER_NIVEAU_VOIE:
        _a_plafonner = (_geo == "Voie") & (appar["niveau_impact"] == NIVEAUX_IMPACTES[0])
        appar.loc[_a_plafonner, "niveau_impact"] = NIVEAUX_IMPACTES[1]
        if _a_plafonner.any():
            print(f"{_a_plafonner.sum()} contrat(s) géocodés à la voie tombaient dans une "
                  f"emprise : ramenés de « certain » à « très probable » — sur un axe de "
                  f"voie, c'est une coïncidence géométrique, pas une preuve.")

    # Le doute ne concerne que les contrats qui auraient été retenus : un contrat
    # à 40 km du feu n'est pas « trop imprécis pour conclure », il est simplement
    # hors sujet. Les basculer tous gonflerait le compteur d'un bruit sans rapport
    # avec le sinistre — et la carte d'autant de marqueurs inutiles.
    _inexploitable = (_geo.isin(NIVEAUX_GEOCODAGE_INEXPLOITABLES)
                      & (appar["niveau_impact"] != "Hors périmètre"))
    appar.loc[_inexploitable, "niveau_impact"] = NIVEAU_INEXPLOITABLE
    if _inexploitable.any():
        _total_mauvais = int(_geo.isin(NIVEAUX_GEOCODAGE_INEXPLOITABLES).sum())
        print(f"{_inexploitable.sum()} contrat(s) au centroïde de commune ou de lieu-dit "
              f"ET dans la zone du feu : sortis du comptage, leur distance à un bâtiment "
              f"ne veut rien dire.")
        print(f"  ({_total_mauvais} contrats sont géocodés à ce niveau sur l'ensemble de "
              f"l'emprise interrogée ; les autres sont hors périmètre, donc hors sujet.)")

appar["est_impacte"] = appar["niveau_impact"].isin(NIVEAUX_IMPACTES)
# On garde le bâti le plus proche quel que soit le seuil (utile au test de sensibilité),
# mais un bâtiment n'est déclaré « touché » que pour les contrats effectivement impactés.
appar["bat_id_brut"] = appar["bat_id"]
appar.loc[~appar["est_impacte"], "bat_id"] = pd.NA

# Un contrat présent sur plusieurs lignes produirait plusieurs points sur la carte
# et plusieurs lignes dans l'export. Ces lignes sont des géocodages concurrents du
# même contrat : on retient donc le plus précis, `level_contrat_mgar` faisant foi.
#
# Ce critère est neutre, et c'est ce qui compte. Départager sur la seule distance au
# bâti reviendrait à retenir systématiquement la position la plus incriminante, donc
# à surestimer l'impact par construction. La distance ne sert plus qu'à départager
# deux géocodages de même niveau.
RANG_NIVEAU_GEOCODAGE = {"Adresse exacte": 0, "Voie": 1, "Lieu-dit": 2,
                         "Centroïde commune": 3, "Non renseigné": 4}

appar["id_plusieurs_positions"] = appar["id"].duplicated(keep=False)

# Deux situations très différentes se cachent sous « doublon », et il faut les
# séparer : à adresse identique le niveau de géocodage tranche sans ambiguïté ;
# à adresses différentes, choisir un niveau revient à choisir un lieu de risque,
# ce qu'aucune règle automatique ne peut fonder. Ces cas sont signalés.
_adresse = (appar["rue_norm"].fillna("") + "|"
            + appar["code_postal_adresse_risque"].astype("string").fillna("") + "|"
            + appar.get("commune_adresse_risque",
                        pd.Series("", index=appar.index)).astype("string").fillna(""))
appar["_n_adresses"] = _adresse.groupby(appar["id"]).transform("nunique")
appar["id_plusieurs_adresses"] = appar["id_plusieurs_positions"] & (appar["_n_adresses"] > 1)

_avant = len(appar)

# Critère 1 — le niveau de géocodage, comme convenu : housenumber avant street,
# avant locality, avant municipality.
appar["_rang_geo"] = (appar["niveau_geocodage"].map(RANG_NIVEAU_GEOCODAGE).fillna(9)
                      if COL_PRECISION else 0)

# Critère 2 — le niveau ne suffit pas : la majorité des contrats en doublon ont
# plusieurs positions AU MÊME niveau, parfois distantes de plusieurs kilomètres.
# On retient alors la position la plus centrale parmi les candidats du meilleur
# niveau. C'est le choix du consensus, qui écarte les valeurs aberrantes — et
# surtout un critère neutre : départager sur la distance au feu retiendrait
# toujours la position la plus incriminante et gonflerait le compte.
_meilleur = appar.groupby("id")["_rang_geo"].transform("min")
_candidat = appar["_rang_geo"] == _meilleur
_x, _y = appar.geometry.x, appar.geometry.y
_cx = _x.where(_candidat).groupby(appar["id"]).transform("mean")
_cy = _y.where(_candidat).groupby(appar["id"]).transform("mean")
appar["_d_centre"] = np.where(_candidat, np.hypot(_x - _cx, _y - _cy), np.inf)

# Étendue des positions candidates : deux géocodages à 5 m l'un de l'autre sont
# sans conséquence, à 3 km la position retenue relève du tirage au sort.
_etendue = (appar.assign(_x=_x, _y=_y).groupby("id")
                 .agg(x0=("_x", "min"), x1=("_x", "max"),
                      y0=("_y", "min"), y1=("_y", "max")))
_etendue["ecart"] = np.hypot(_etendue.x1 - _etendue.x0, _etendue.y1 - _etendue.y0)
appar["ecart_positions_m"] = appar["id"].map(_etendue["ecart"]).round(1)
appar["position_incertaine"] = appar["ecart_positions_m"] > SEUIL_ECART_POSITIONS_M

appar = (appar.sort_values(["_rang_geo", "_d_centre"], na_position="last")
              .drop_duplicates(subset="id", keep="first")
              .sort_index()
              .drop(columns=["_rang_geo", "_d_centre", "_n_adresses"], errors="ignore"))

if _avant != len(appar):
    _multi = appar["id_plusieurs_positions"].sum()
    _multi_adr = appar["id_plusieurs_adresses"].sum()
    print(f"{_avant - len(appar)} ligne(s) écartée(s) sur {_multi} contrat(s) à "
          f"géocodages concurrents.")
    print(f"  {_multi - _multi_adr} à adresse identique — le niveau de géocodage tranche ;")
    print(f"  {_multi_adr} à adresses différentes — plusieurs lieux de risque possibles, "
          f"marqués `id_plusieurs_adresses` : à instruire, la règle ne fait qu'un choix "
          f"par défaut.")
    if COL_PRECISION:
        retenus = appar.loc[appar["id_plusieurs_positions"], "niveau_geocodage"]
        print("  niveau retenu pour ces contrats :")
        for niv, nb in retenus.value_counts().items():
            print(f"    {niv:<20} {nb}")
    _incertains = appar.loc[appar["id_plusieurs_positions"], "ecart_positions_m"]
    _loin = int((_incertains > SEUIL_ECART_POSITIONS_M).sum())
    print(f"  écart entre positions candidates : médiane {_incertains.median():.0f} m, "
          f"max {_incertains.max():.0f} m")
    print(f"  {_loin} contrat(s) au-delà de {SEUIL_ECART_POSITIONS_M:.0f} m d'écart : la "
          f"position retenue y est un choix par défaut, pas une localisation fiable "
          f"— marqués `position_incertaine` dans l'export.")
    garde = appar.loc[appar["id_plusieurs_positions"] & appar["est_impacte"]]
    print(f"  dont {len(garde)} contrat(s) impacté(s) issus d'un identifiant à plusieurs "
          f"positions — à vérifier avant tout contact.")

ORDRE_NIVEAUX = ["Certain — dans l'emprise brûlée", "Très probable — à moins de 10 m",
                 "Probable — à moins de 25 m", "Exposé — dans le périmètre du feu",
                 NIVEAU_INEXPLOITABLE, "Hors périmètre"]
appar["niveau_impact"] = pd.Categorical(appar["niveau_impact"], ORDRE_NIVEAUX, ordered=True)

display(appar["niveau_impact"].value_counts().sort_index().rename("contrats").to_frame())

## 8. Réponse chiffrée

« Combien ? » se décline en trois compteurs différents, et **il faut les trois** :

* **bâtiments distincts** touchés portant au moins un contrat — la réponse littérale
  à la question, dédoublonnée (un immeuble ne compte qu'une fois) ;
* **contrats** impactés — la charge de gestion réelle ;
* **sociétaires distincts** — le nombre de foyers à contacter.

In [ ]:
cible = appar[appar["dans_perimetre_demande"]]          # RP / RS / PNO
impact = cible[cible["est_impacte"]]

KPI = {
    "batis_touches": int(impact["bat_id"].nunique()),
    "contrats": int(impact["id"].nunique()),
    "societaires": int(impact["id_societaire"].nunique()),
    "certains": int((impact["niveau_impact"] == "Certain — dans l'emprise brûlée").sum()),
    "en_perimetre": int((cible["niveau_impact"] == "Exposé — dans le périmètre du feu").sum()),
    "hors_cible_impactes": int(appar.loc[~appar["dans_perimetre_demande"]
                                         & appar["est_impacte"], "id"].nunique()),
    "suspects": int(impact["geocodage_suspect"].sum()),
    "inexploitables": int((cible["niveau_impact"] == NIVEAU_INEXPLOITABLE).sum()),
    "declares": int(impact["sinistre_declare"].sum()),
    "impactes_sans_declaration": int((~impact["sinistre_declare"]).sum()),
    "declares_non_detectes": int(cible.loc[~cible["est_impacte"], "sinistre_declare"].sum()),
}
for k, v in KPI.items():
    print(f"{k:>22} : {v:>6,}".replace(",", " "))

In [ ]:
# --- Tableau 1 : synthèse par feu et par segment --------------------------- #
synthese = (impact.groupby(["feu_rattache", "segment"], observed=True)
                  .agg(batis_touches=("bat_id", "nunique"),
                       contrats=("id", "nunique"),
                       societaires=("id_societaire", "nunique"))
                  .reset_index()
                  .rename(columns={"feu_rattache": "feu"}))
synthese["libelle"] = synthese["segment"].map(LIB_SEGMENT)
synthese = synthese.sort_values(["feu", "contrats"], ascending=[True, False])
display(synthese[["feu", "segment", "libelle", "batis_touches", "contrats", "societaires"]])

# --- Tableau 2 : détail par niveau de certitude ---------------------------- #
detail = (pd.crosstab(cible["feu_rattache"].fillna("Aucun feu à proximité"),
                      cible["niveau_impact"], dropna=False)
            .reindex(columns=ORDRE_NIVEAUX, fill_value=0))
detail.index.name = "Feu le plus proche"
display(detail)

In [ ]:
# --- Tableau 3 : sensibilité au seuil de distance -------------------------- #
# Les positions trop imprécises sont exclues : leur distance n'est pas interprétable.
cible_exploitable = cible[cible["niveau_impact"] != NIVEAU_INEXPLOITABLE]

lignes = []
for s in SEUILS_SENSIBILITE_M:
    sel = cible_exploitable[cible_exploitable["distance_bati_m"].le(s)]
    # `bat_id` a été neutralisé au-delà du seuil retenu : on le relit sur l'appariement brut
    bat_s = appar.loc[sel.index, "bat_id_brut"]
    lignes.append({"seuil_m": s,
                   "batis_touches": int(bat_s.nunique()),
                   "contrats": int(sel["id"].nunique()),
                   "societaires": int(sel["id_societaire"].nunique())})
sensibilite = pd.DataFrame(lignes)
sensibilite["retenu"] = np.where(sensibilite["seuil_m"] == SEUIL_PROBABLE_M, "◀ retenu", "")
display(sensibilite)

In [ ]:
# --- Matrice de validation : géométrie × sinistre déclaré ------------------ #
# Le seul contrôle externe dont on dispose. Il mesure la méthode au lieu de la
# supposer juste, et sort la liste des impactés qui n'ont pas encore déclaré.
matrice = (pd.crosstab(cible["niveau_impact"], cible["sinistre_declare"],
                       dropna=False)
             .reindex(index=ORDRE_NIVEAUX, fill_value=0))
matrice.columns = [{False: "Sans déclaration", True: "Sinistre ouvert"}.get(c, c)
                   for c in matrice.columns]
for _c in ("Sinistre ouvert", "Sans déclaration"):
    if _c not in matrice:
        matrice[_c] = 0
matrice["Total"] = matrice.sum(axis=1)
matrice["Taux de déclaration"] = (matrice["Sinistre ouvert"] / matrice["Total"]
                                  .replace(0, np.nan)).map(
    lambda v: "—" if pd.isna(v) else f"{v:.0%}")
display(matrice)

print(f"\nParmi les {KPI['contrats']} contrats retenus comme impactés, "
      f"{KPI['declares']} ont déjà un sinistre ouvert et "
      f"{KPI['impactes_sans_declaration']} n'ont rien déclaré → liste de contact prioritaire.")
print(f"{KPI['declares_non_detectes']} contrat(s) déclarent un sinistre sans être retenus "
      f"par la géométrie : autant de cas où la méthode passe à côté, à examiner.")

# --- Tableau 4 : qualité de l'assuré et nature du bien --------------------- #
if "qualite" in impact:
    display(pd.crosstab(impact["qualite"], impact["segment"],
                        margins=True, margins_name="Total"))
if "type_bien" in impact:
    display(pd.crosstab(impact["type_bien"], impact["segment"],
                        margins=True, margins_name="Total"))

# --- Export gestion --------------------------------------------------------- #
cols_export = [c for c in [
    "id_societaire", "numero_intercalaire", "id", "numero_contrat", "segment",
    "qualite", "type_bien", "niveau_geocodage",
    "id_plusieurs_positions", "id_plusieurs_adresses",
    "position_incertaine", "ecart_positions_m",
    "sinistre_declare", "nb_sinistres_ouverts", "numeros_sinistres", "date_declaration",
    "rue_adresse_risque",
    "code_postal_adresse_risque", "commune_adresse_risque",
    "lon_contrat_mgar", "lat_contrat_mgar", "feu_rattache", "niveau_impact",
    "distance_bati_m", "bat_id", "surface_bati_m2", "surface_habitable",
    "nombre_pieces_totales", "montant_capital_mobilier", "geocodage_suspect",
] if c in impact.columns]

export = (impact[cols_export]
          .sort_values(["feu_rattache", "niveau_impact", "distance_bati_m"]))
export.to_csv(FICHIER_CSV, index=False, encoding="utf-8-sig")
print(f"{len(export)} contrats exportés → {FICHIER_CSV}")
export.head(10)

## 9. Carte interactive

Encodage : la **couleur porte le segment** (RP / RS / PNO — trois teintes validées
pour la lisibilité en vision des couleurs déficiente), la **taille et l'opacité
portent le niveau de certitude**. Chaque niveau est une couche activable
séparément, pour isoler les cas certains.

In [ ]:
PALETTE_SEGMENT = {"RP": "#2a78d6", "RS": "#eb6834", "PNO": "#1baf7a", "Autres": "#8a8983"}
STYLE_NIVEAU = {
    "Certain — dans l'emprise brûlée": dict(radius=7.5, fill_opacity=0.95, weight=2.0),
    "Très probable — à moins de 10 m":   dict(radius=6.0, fill_opacity=0.75, weight=1.5),
    "Probable — à moins de 25 m":        dict(radius=5.0, fill_opacity=0.50, weight=1.2),
    "Exposé — dans le périmètre du feu": dict(radius=3.5, fill_opacity=0.25, weight=0.8),
    NIVEAU_INEXPLOITABLE: dict(radius=4.0, fill_opacity=0.30, weight=1.0),
}

carte_pts = appar.to_crs(CRS_AFFICHAGE)
contours_wgs = contours.to_crs(CRS_AFFICHAGE)
centre = contours_wgs.geometry.union_all().centroid

m = folium.Map(location=[centre.y, centre.x], tiles=FOND_DE_CARTE, control_scale=True)
# Les deux feux sont distants de ~60 km : on cadre sur leur emprise commune
# plutôt que sur un niveau de zoom fixe, sinon Biscarrosse sort de l'écran.
x0, y0, x1, y1 = contours_wgs.total_bounds
m.fit_bounds([[y0, x0], [y1, x1]], padding=(20, 20))

# Périmètres des feux
folium.GeoJson(
    contours_wgs.assign(geometry=contours_wgs.geometry.set_precision(1e-6)),
    name="Périmètre des feux",
    style_function=lambda _: {"color": "#e34948", "weight": 2.5,
                              "fillColor": "#e34948", "fillOpacity": 0.06},
    tooltip=folium.GeoJsonTooltip(fields=["feu", "surface_feu_ha"],
                                  aliases=["Feu", "Surface (ha)"]),
).add_to(m)

# Bâtiments relevés dans l'emprise brûlée.
# Les emprises sont simplifiées de 30 cm et les coordonnées arrondies au décimètre
# avant sérialisation : à 2 667 polygones, les 15 décimales par défaut pèsent
# plusieurs mégaoctets pour une précision sans objet à l'écran.
batis_carte = batis.copy()
batis_carte["geometry"] = batis_carte.geometry.simplify(0.3, preserve_topology=True)
batis_carte = batis_carte.to_crs(CRS_AFFICHAGE)
batis_carte["geometry"] = batis_carte.geometry.set_precision(1e-6)

folium.GeoJson(
    batis_carte,
    name=f"Bâtiments dans l'emprise brûlée ({len(batis_carte)})",
    style_function=lambda _: {"color": "#0d366b", "weight": 0.6,
                              "fillColor": "#1f2937", "fillOpacity": 0.85},
    tooltip=folium.GeoJsonTooltip(fields=["feu", "bat_id", "surface_bati_m2"],
                                  aliases=["Feu", "Bâtiment", "Emprise (m²)"]),
).add_to(m)

# --------------------------------------------------------------------------- #
# Deux traitements selon l'usage de la couche.
#
# Les trois niveaux retenus comme impactés sont peu nombreux et servent à la
# gestion : marqueurs individuels, survol identifiant, fiche complète au clic.
#
# Les couches de contexte — exposés, positions imprécises, hors périmètre — se
# comptent par milliers et ne servent qu'à situer. Un marqueur individuel avec
# fiche y pèse ~2,5 Ko : à 15 000 points, cela ajoute 35 Mo au livrable. Elles
# sont donc rendues en une seule couche GeoJSON, avec le survol seul.
# --------------------------------------------------------------------------- #
def _jour(v):
    """Date lisible, quelle que soit la forme reçue (Timestamp, str, NaT)."""
    if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
        return "—"
    try:
        return pd.to_datetime(v).strftime("%d/%m/%Y")
    except Exception:                                            # noqa: BLE001
        return str(v)[:10]


for niveau_lbl in NIVEAUX_IMPACTES:
    style = STYLE_NIVEAU[niveau_lbl]
    sel = carte_pts[(carte_pts["niveau_impact"] == niveau_lbl)
                    & carte_pts["dans_perimetre_demande"]]
    if sel.empty:
        continue
    fg = folium.FeatureGroup(name=f"{niveau_lbl} ({len(sel)})", show=True)
    for r in sel.itertuples():
        couleur = PALETTE_SEGMENT.get(r.segment, PALETTE_SEGMENT["Autres"])
        if r.niveau_impact == NIVEAU_INEXPLOITABLE:
            couleur = PALETTE_SEGMENT["Autres"]      # gris : position non fiable
        statut = getattr(r, "statut", "Autre / non renseigné")
        dist = "—" if pd.isna(r.distance_bati_m) else f"{r.distance_bati_m:.1f} m"
        adresse = " ".join(str(getattr(r, c, "") or "") for c in
                           ("rue_adresse_risque", "code_postal_adresse_risque",
                            "commune_adresse_risque")).strip()

        # Survol : l'identification, et rien d'autre — c'est ce qu'on cherche en
        # balayant la carte pour rapprocher un point d'un dossier.
        survol = (f'<div style="font:12px/1.55 system-ui,-apple-system,sans-serif;">'
                  f'<b>Sociétaire {r.id_societaire}</b><br>'
                  f'Intercalaire {r.numero_intercalaire}<br>'
                  f'<span style="color:{couleur};">●</span> {r.segment} · {statut}'
                  + ("<br>🔴 <b>sinistre ouvert</b>"
                     if getattr(r, "sinistre_declare", False) else "")
                  + '</div>')

        # Clic : la fiche complète.
        fiche = (f"<div style='font:12.5px/1.6 system-ui,-apple-system,sans-serif;'>"
                 f"<b>{LIB_SEGMENT.get(r.segment, r.segment)}</b><hr style='margin:5px 0'>"
                 f"<b>Sociétaire :</b> {r.id_societaire}<br>"
                 f"<b>Intercalaire :</b> {r.numero_intercalaire}<br>"
                 f"<b>Clé contrat :</b> {r.id}<br>"
                 f"<b>Statut :</b> {getattr(r, 'qualite', statut)}<br>"
                 f"{adresse}<hr style='margin:5px 0'>"
                 f"Niveau : {r.niveau_impact}<br>"
                 f"Distance au bâti de l'emprise : {dist}"
                 + ("<br><i>⚠️ géocodage à confirmer</i>" if r.geocodage_suspect else "")
                 + (f"<hr style='margin:5px 0'><b style='color:#b91c1c'>▲ Sinistre "
                    f"ouvert</b><br>N° : {getattr(r, 'numeros_sinistres', '—')}<br>"
                    f"Ouvert le : {_jour(getattr(r, 'date_declaration', None))}"
                    + (f"<br>({int(r.nb_sinistres_ouverts)} dossiers)"
                       if getattr(r, "nb_sinistres_ouverts", 0) > 1 else "")
                    if getattr(r, "sinistre_declare", False)
                    else "<hr style='margin:5px 0'><i>Aucune déclaration à ce jour</i>")
                 + "</div>")

        # La couleur porte déjà le segment et la taille le niveau de certitude :
        # le statut d'occupation passe donc par la forme, pas par une 2ᵉ palette.
        # Disque plein = propriétaire, anneau = locataire, contour gris = autre.
        if statut == "Locataire":
            forme = dict(color=couleur, fillColor="#ffffff", fillOpacity=0.9,
                         weight=max(2.2, style["weight"] + 0.8))
        elif statut == "Propriétaire":
            forme = dict(color="#ffffff", fillColor=couleur,
                         fillOpacity=style["fill_opacity"], weight=style["weight"])
        else:
            forme = dict(color="#52514e", fillColor=couleur,
                         fillOpacity=style["fill_opacity"] * 0.6,
                         weight=max(1.2, style["weight"]))

        # Un sinistre déclaré est le signal le plus fort de la carte : il change la
        # forme du marqueur, pas seulement sa couleur. Triangle = sinistre ouvert,
        # cercle = aucune déclaration. Le plein / l'anneau continuent de porter le
        # statut d'occupation, un triangle évidé restant un triangle.
        if getattr(r, "sinistre_declare", False):
            cote = style["radius"] * 2 + 5
            folium.Marker(
                location=[r.geometry.y, r.geometry.x],
                icon=folium.DivIcon(
                    html=(f'<svg width="{cote:.0f}" height="{cote:.0f}" viewBox="0 0 20 20">'
                          f'<polygon points="10,1.6 18.6,17.4 1.4,17.4" '
                          f'fill="{forme["fillColor"]}" fill-opacity="{forme["fillOpacity"]}" '
                          f'stroke="{forme["color"]}" stroke-width="{forme["weight"] + 0.4:.1f}" '
                          f'stroke-linejoin="round"/></svg>'),
                    icon_size=(cote, cote), icon_anchor=(cote / 2, cote / 2), class_name=""),
                tooltip=folium.Tooltip(survol, sticky=True),
                popup=folium.Popup(fiche, max_width=340),
            ).add_to(fg)
        else:
            folium.CircleMarker(
                location=[r.geometry.y, r.geometry.x], radius=style["radius"], fill=True,
                tooltip=folium.Tooltip(survol, sticky=True),
                popup=folium.Popup(fiche, max_width=340), **forme,
            ).add_to(fg)
    fg.add_to(m)

# --------------------------------------------------------------------------- #
# Le dénominateur : l'emprise interrogée et les contrats examinés mais non retenus.
# Sans eux, la carte ne montre que des impacts et ne dit rien de ce qui a été
# regardé. Rendus en une seule couche GeoJSON — à 15 000 points, autant de
# marqueurs individuels alourdiraient le fichier d'un ordre de grandeur.
# --------------------------------------------------------------------------- #
fg_zone = folium.FeatureGroup(name="Emprise interrogée (requête)", show=True)
folium.Rectangle(
    bounds=[[LAT_MIN, LON_MIN], [LAT_MAX, LON_MAX]],
    color="#52514e", weight=1.5, dash_array="7,6", fill=False,
    tooltip=("Emprise du pré-filtre de la requête — "
             f"lon [{LON_MIN}, {LON_MAX}] / lat [{LAT_MIN}, {LAT_MAX}]"),
).add_to(fg_zone)
fg_zone.add_to(m)

CHAMPS_CONTEXTE = ["id_societaire", "numero_intercalaire", "segment"]
ALIAS_CONTEXTE = ["Sociétaire", "Intercalaire", "Segment"]


def couche_contexte(sel, nom, rayon, opacite, en_gris=False, identifier=True):
    """Couche légère : une seule structure GeoJSON, survol seul, pas de fiche.

    `identifier=False` ne garde que le segment dans l'infobulle. Chaque propriété
    est répétée par point, et le GeoJSON étant échappé pour tenir dans l'attribut
    srcdoc de l'iframe, chaque guillemet y compte pour six caractères. Sur une
    couche de plusieurs milliers de points, l'identification pèse plus lourd que
    les géométries — on ne la garde donc que là où elle sert à travailler.
    """
    if sel.empty:
        return
    if len(sel) > MAX_POINTS_CONTEXTE:
        print(f"⚠️  {nom} : {len(sel)} contrats, la carte n'en affiche que "
              f"{MAX_POINTS_CONTEXTE} (MAX_POINTS_CONTEXTE). Affichage échantillonné, "
              f"comptages inchangés.")
        sel = sel.sample(MAX_POINTS_CONTEXTE, random_state=0)

    voulus = CHAMPS_CONTEXTE if identifier else ["segment"]
    champs = [c for c in voulus if c in sel.columns]
    alias = [a for c, a in zip(CHAMPS_CONTEXTE, ALIAS_CONTEXTE)
             if c in champs and c in sel.columns]
    # 15 décimales par coordonnée pèsent lourd à cette volumétrie pour une précision
    # absurde ; 5 décimales valent environ 1 m, largement assez ici.
    sel = sel.assign(geometry=sel.geometry.apply(
        lambda g: Point(round(g.x, 5), round(g.y, 5))))

    folium.GeoJson(
        sel[champs + ["geometry"]],
        name=f"{nom} ({len(sel)})",
        show=False,
        marker=folium.CircleMarker(radius=rayon, fill=True),
        style_function=lambda f: {
            "radius": rayon, "weight": 0.5, "color": "#ffffff", "fillOpacity": opacite,
            "fillColor": (PALETTE_SEGMENT["Autres"] if en_gris else
                          PALETTE_SEGMENT.get(f["properties"].get("segment"),
                                              PALETTE_SEGMENT["Autres"])),
        },
        tooltip=folium.GeoJsonTooltip(fields=champs, aliases=alias),
    ).add_to(m)


_cible_carte = carte_pts[carte_pts["dans_perimetre_demande"]]
for _niveau, _rayon, _op, _gris, _ident in (
    # exposés et positions douteuses restent identifiables : on travaille dessus
    ("Exposé — dans le périmètre du feu", 3.5, 0.45, False, True),
    (NIVEAU_INEXPLOITABLE, 3.5, 0.40, True, True),
    # hors périmètre : repère de volume, l'identification n'y sert à rien
    ("Hors périmètre", 2.5, 0.55, False, False),
):
    couche_contexte(_cible_carte[_cible_carte["niveau_impact"] == _niveau],
                    _niveau, _rayon, _op, _gris, _ident)

folium.LayerControl(position="topright", collapsed=False).add_to(m)

# Raccourcis de cadrage. À l'échelle qui couvre les deux feux (60 km d'écart), un
# bâtiment de 137 m² occupe moins d'un pixel : la vue d'ensemble sert à situer, pas
# à lire. Ces liens amènent directement à l'échelle où le bâti devient lisible.
zones = {"Vue d'ensemble": [[y0, x0], [y1, x1]]}
for nom_feu in FEUX:
    fx0, fy0, fx1, fy1 = contours_wgs.loc[contours_wgs.feu == nom_feu].total_bounds
    zones[f"Feu de {nom_feu}"] = [[fy0, fx0], [fy1, fx1]]

m.get_root().script.add_child(folium.Element(f"""
  window.addEventListener("load", function () {{
    var zones = {json.dumps(zones)};
    var ctlZones = L.control({{position: 'topleft'}});
    ctlZones.onAdd = function () {{
        var d = L.DomUtil.create('div', 'leaflet-bar');
        d.style.cssText = 'background:#fff;padding:7px 9px;font:12px/1.65 ' +
            'system-ui,-apple-system,sans-serif;color:#0b0b0b;';
        var h = '<div style="font-weight:650;margin-bottom:2px;">Aller à</div>';
        Object.keys(zones).forEach(function (k) {{
            h += '<a href="#" data-z="' + k + '" style="display:block;color:#1c5cab;' +
                 'text-decoration:none;white-space:nowrap;">' + k + '</a>';
        }});
        d.innerHTML = h;
        L.DomEvent.disableClickPropagation(d);
        d.querySelectorAll('a').forEach(function (a) {{
            a.onclick = function (e) {{
                e.preventDefault();
                {m.get_name()}.fitBounds(zones[a.getAttribute('data-z')]);
            }};
        }});
        return d;
    }};
    ctlZones.addTo({m.get_name()});
  }});
"""))

LEGENDE = """
<div style="position:fixed;bottom:24px;right:12px;z-index:9999;background:rgba(255,255,255,.94);
  padding:9px 11px;border-radius:9px;box-shadow:0 1px 8px rgba(0,0,0,.28);max-width:225px;
  font:11.5px/1.35 system-ui,-apple-system,'Segoe UI',Roboto,sans-serif;color:#0b0b0b;">
  <div style="font-weight:650;margin-bottom:6px;">Segment</div>
  <div><span style="display:inline-block;width:11px;height:11px;border-radius:50%;
    background:#2a78d6;margin-right:6px;"></span>RP — rés. principale</div>
  <div><span style="display:inline-block;width:11px;height:11px;border-radius:50%;
    background:#eb6834;margin-right:6px;"></span>RS — rés. secondaire</div>
  <div><span style="display:inline-block;width:11px;height:11px;border-radius:50%;
    background:#1baf7a;margin-right:6px;"></span>PNO — propr. non occupant</div>
  <div style="font-weight:650;margin:7px 0 4px;">Occupation</div>
  <div><span style="display:inline-block;width:11px;height:11px;border-radius:50%;
    background:#2a78d6;border:1.5px solid #fff;margin-right:6px;
    vertical-align:-1px;"></span>Propriétaire — plein</div>
  <div><span style="display:inline-block;width:11px;height:11px;border-radius:50%;
    background:#fff;border:2px solid #2a78d6;margin-right:6px;
    vertical-align:-1px;"></span>Locataire — anneau</div>
  <div><span style="display:inline-block;width:11px;height:11px;border-radius:50%;
    background:#2a78d6;border:1.5px solid #52514e;margin-right:6px;
    vertical-align:-1px;opacity:.6;"></span>Autre — contour gris</div>
  <div style="font-weight:650;margin:7px 0 4px;">Sinistre</div>
  <div><span style="display:inline-block;width:0;height:0;border-left:6px solid transparent;
    border-right:6px solid transparent;border-bottom:10px solid #2a78d6;margin-right:6px;
    vertical-align:0px;"></span>sinistre incendie ouvert</div>
  <div><span style="display:inline-block;width:11px;height:11px;border-radius:50%;
    background:#2a78d6;margin-right:6px;vertical-align:-1px;"></span>aucune déclaration</div>
  <div style="font-weight:650;margin:7px 0 4px;">Certitude</div>
  <div>● grand — dans l'emprise brûlée</div>
  <div>● moyen — &lt; 10 m</div>
  <div>● petit — &lt; 25 m</div>
  <div style="margin-top:7px;"><span style="display:inline-block;width:11px;height:11px;
    background:#1f2937;margin-right:6px;"></span>Bâti de l'emprise brûlée</div>
  <div style="margin-top:5px;"><span style="display:inline-block;width:16px;
    border-top:1.5px dashed #52514e;margin-right:6px;vertical-align:3px;"></span>Emprise interrogée</div>
  <div style="margin-top:8px;padding-top:7px;border-top:1px solid #e0dfda;color:#52514e;">
    Survol : sociétaire + intercalaire<br>Clic : fiche complète</div>
</div>"""
m.get_root().html.add_child(folium.Element(LEGENDE))


def rendre_carte_autonome(carte: folium.Map) -> str:
    """Rend la carte en HTML sans aucune dépendance CDN.

    folium référence Leaflet, jQuery, Bootstrap et awesome-markers via des <script src>
    et <link href> distants. Sur un poste dont le proxy bloque ces CDN, la page reste
    blanche (incident déjà rencontré sur les cartes de grêle). On retire donc toutes
    les ressources externes et on réinjecte Leaflet — la seule réellement nécessaire —
    depuis `incendie/assets/`.
    """
    import re

    doc = carte.get_root().render()
    if not LEAFLET_EMBARQUE:
        print("⚠️  incendie/assets/leaflet.js absent → la carte dépendra des CDN.")
        return doc

    doc = re.sub(r'<script[^>]+src="https?://[^"]+"[^>]*>\s*</script>', "", doc)
    doc = re.sub(r'<link[^>]+href="https?://[^"]+"[^>]*/?>', "", doc)

    def _lire(nom):
        return (DOSSIER_ASSETS / nom).read_text(encoding="utf-8").replace("</script>", r"<\/script>")

    # jQuery avant Leaflet : folium construit chaque popup avec `$(...)`, et une
    # ReferenceError sur `$` interrompt tout le script — y compris les marqueurs
    # et le sélecteur de couches déclarés plus loin.
    inject = (f"<style>{(DOSSIER_ASSETS / 'leaflet.css').read_text(encoding='utf-8')}</style>\n"
              f"<script>{_lire('jquery.js')}</script>\n"
              f"<script>{_lire('leaflet.js')}</script>\n")
    return doc.replace("</head>", inject + "</head>", 1)


CARTE_HTML = rendre_carte_autonome(m)
print(f"Carte rendue — {len(CARTE_HTML) / 1024:.0f} Ko, "
      f"Leaflet {'embarqué' if LEAFLET_EMBARQUE else 'via CDN'}")
m

## 10. Génération du livrable HTML *one shot*

Un fichier unique : compteurs, méthode, tableaux et carte embarquée. Il s'ouvre
dans n'importe quel navigateur et se transmet tel quel.

In [ ]:
CSS = """
:root{color-scheme:light dark}
*{box-sizing:border-box}
body{margin:0;font:15px/1.6 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;
  background:var(--bg);color:var(--txt)}
:root{--bg:#f4f4f2;--surface:#fcfcfb;--txt:#0b0b0b;--txt2:#52514e;--line:#e0dfda;
  --rp:#2a78d6;--rs:#eb6834;--pno:#1baf7a;--alert:#e34948}
@media (prefers-color-scheme:dark){:root:where(:not([data-theme="light"])){
  --bg:#111110;--surface:#1a1a19;--txt:#fff;--txt2:#c3c2b7;--line:#383835;
  --rp:#3987e5;--rs:#d95926;--pno:#199e70;--alert:#e66767}}
:root[data-theme="dark"]{--bg:#111110;--surface:#1a1a19;--txt:#fff;--txt2:#c3c2b7;
  --line:#383835;--rp:#3987e5;--rs:#d95926;--pno:#199e70;--alert:#e66767}
.wrap{max-width:1180px;margin:0 auto;padding:2rem 1.25rem 4rem}
header h1{font-size:1.6rem;margin:0 0 .3rem;letter-spacing:-.01em}
header p{margin:0;color:var(--txt2);font-size:.93rem}
.bandeau{margin:1.25rem 0;padding:.8rem 1rem;border-radius:10px;font-size:.88rem;
  background:#fff7ed;border:1px solid #fed7aa;color:#9a3412}
@media (prefers-color-scheme:dark){:root:where(:not([data-theme="light"])) .bandeau{
  background:#2a1a0d;border-color:#7c3d12;color:#fdba74}}
section{background:var(--surface);border:1px solid var(--line);border-radius:14px;
  padding:1.4rem 1.5rem;margin:1.25rem 0}
h2{font-size:1.05rem;margin:0 0 1rem;letter-spacing:-.005em}
h2 .n{color:var(--txt2);font-weight:400;margin-right:.45rem}
.kpis{display:grid;gap:.9rem;grid-template-columns:repeat(auto-fit,minmax(190px,1fr))}
.kpi{border:1px solid var(--line);border-radius:12px;padding:1rem 1.1rem;background:var(--bg)}
.kpi .v{font-size:2.1rem;font-weight:640;line-height:1.05;letter-spacing:-.02em;
  font-variant-numeric:tabular-nums}
.kpi .l{font-size:.8rem;color:var(--txt2);margin-top:.35rem;line-height:1.4}
.kpi.lead .v{color:var(--alert)}
.tbl{overflow-x:auto;-webkit-overflow-scrolling:touch}
table{border-collapse:collapse;width:100%;font-size:.88rem;min-width:520px}
th,td{padding:.55rem .7rem;text-align:right;border-bottom:1px solid var(--line);
  font-variant-numeric:tabular-nums;white-space:nowrap}
th:first-child,td:first-child,th.t,td.t{text-align:left;font-variant-numeric:normal}
thead th{color:var(--txt2);font-weight:600;font-size:.8rem;text-transform:uppercase;
  letter-spacing:.03em;border-bottom:1.5px solid var(--line)}
tbody tr:last-child td{border-bottom:none}
tr.tot td{font-weight:650;border-top:1.5px solid var(--line)}
.pill{display:inline-flex;align-items:center;gap:.4rem;font-weight:600}
.dot{width:9px;height:9px;border-radius:50%;flex:none}
.map{height:780px;border:1px solid var(--line);border-radius:12px;overflow:hidden}
.map iframe{width:100%;height:100%;border:0;display:block}
.notes{font-size:.87rem;color:var(--txt2)}
.notes li{margin-bottom:.5rem}
footer{margin-top:2rem;font-size:.78rem;color:var(--txt2);text-align:center}
"""


def _n(v):
    return f"{int(v):,}".replace(",", " ")


def _tab(df, cls_first=True):
    th = "".join(f"<th{' class=t' if i == 0 and cls_first else ''}>{_html.escape(str(c))}</th>"
                 for i, c in enumerate(df.columns))
    tr = ""
    for _, r in df.iterrows():
        tds = "".join(
            f"<td{' class=t' if i == 0 and cls_first else ''}>"
            f"{_n(v) if isinstance(v, (int, np.integer)) else _html.escape(str(v))}</td>"
            for i, v in enumerate(r))
        tr += f"<tr>{tds}</tr>"
    return f'<div class="tbl"><table><thead><tr>{th}</tr></thead><tbody>{tr}</tbody></table></div>'


# --- KPI par segment -------------------------------------------------------- #
par_seg = (impact.groupby("segment", observed=True)
                 .agg(batis=("bat_id", "nunique"), contrats=("id", "nunique"),
                      societaires=("id_societaire", "nunique"))
                 .reindex(SEGMENTS_CIBLE).fillna(0).astype(int))

kpi_html = f"""
<div class="kpi lead"><div class="v">{_n(KPI['batis_touches'])}</div>
  <div class="l">bâtiments distincts concernés<br>portant un contrat RP / RS / PNO</div></div>
<div class="kpi"><div class="v">{_n(KPI['contrats'])}</div>
  <div class="l">contrats habitation impactés</div></div>
<div class="kpi"><div class="v">{_n(KPI['societaires'])}</div>
  <div class="l">sociétaires distincts à contacter</div></div>
<div class="kpi"><div class="v">{_n(KPI['certains'])}</div>
  <div class="l">dont le point contrat tombe<br><b>dans</b> l'emprise brûlée</div></div>
<div class="kpi"><div class="v">{_n(KPI['en_perimetre'])}</div>
  <div class="l">dans le périmètre du feu mais<br>hors emprise bâtie — exposés</div></div>
<div class="kpi"><div class="v">{_n(KPI['inexploitables'])}</div>
  <div class="l">position trop imprécise<br>pour conclure — à instruire</div></div>
"""

seg_rows = "".join(
    f'<tr><td class=t><span class="pill"><span class="dot" style="background:'
    f'{PALETTE_SEGMENT[s]}"></span>{s} — {LIB_SEGMENT[s]}</span></td>'
    f"<td>{_n(par_seg.loc[s, 'batis'])}</td><td>{_n(par_seg.loc[s, 'contrats'])}</td>"
    f"<td>{_n(par_seg.loc[s, 'societaires'])}</td></tr>" for s in SEGMENTS_CIBLE)
seg_rows += (f'<tr class=tot><td class=t>Total RP + RS + PNO</td>'
             f"<td>{_n(KPI['batis_touches'])}</td><td>{_n(KPI['contrats'])}</td>"
             f"<td>{_n(KPI['societaires'])}</td></tr>")

# --- Croisement statut d'occupation × segment ------------------------------- #
MARQUE_STATUT = {
    "Propriétaire": "background:#52514e;border:1.5px solid #fff;",
    "Locataire": "background:#fff;border:2px solid #52514e;",
    "Autre / non renseigné": "background:#52514e;border:1.5px solid #52514e;opacity:.6;",
}
croise = (pd.crosstab(impact["statut"], impact["segment"])
            .reindex(index=ORDRE_STATUTS, columns=list(SEGMENTS_CIBLE))
            .fillna(0).astype(int))
croise["Total"] = croise.sum(axis=1)

stat_rows = ""
for s in ORDRE_STATUTS:
    cells = "".join(f"<td>{_n(croise.loc[s, c])}</td>"
                    for c in list(SEGMENTS_CIBLE) + ["Total"])
    stat_rows += (f'<tr><td class=t><span class="pill"><span style="width:11px;height:11px;'
                  f'border-radius:50%;display:inline-block;{MARQUE_STATUT[s]}"></span>'
                  f"{s}</span></td>{cells}</tr>")
tot = croise.sum()
stat_rows += ('<tr class=tot><td class=t>Total</td>'
              + "".join(f"<td>{_n(tot[c])}</td>" for c in list(SEGMENTS_CIBLE) + ["Total"])
              + "</tr>")

nb_proprio = int(croise.loc["Propriétaire", "Total"])
nb_loc = int(croise.loc["Locataire", "Total"])
pno_loc = int(croise.loc["Locataire", "PNO"]) if "PNO" in croise.columns else 0

bandeau = ""
if MODE_SOURCE == "simulation":
    bandeau = ('<div class="bandeau"><b>⚠️ Données sociétaires SIMULÉES.</b> '
               "Aucun accès BigQuery ni export local n'a été trouvé : les chiffres et les "
               "points ci-dessous sont un jeu de test destiné à valider la chaîne de "
               "traitement. Relancer le notebook avec accès à "
               "<code>contrat_mgar_gps_iris</code> pour obtenir les résultats réels. "
               "Les données incendie, elles, sont bien les données réelles.</div>")

if COL_PRECISION:
    texte_geocodage = (
        f"{_n(KPI['suspects'])} contrat(s) impacté(s) ont une position qui n'est pas "
        f"posée sur l'adresse — centroïde de commune ou lieu-dit — d'après la colonne "
        f"<code>{COL_PRECISION}</code> de la table de géocodage. Leur rattachement à un "
        f"bâtiment n'est pas fiable et demande une vérification avant tout contact. "
        f"Le niveau de géocodage de chaque contrat figure en colonne "
        f"<code>niveau_geocodage</code> de l'export CSV.")
else:
    texte_geocodage = (
        f"{_n(KPI['suspects'])} contrat(s) impacté(s) partagent leurs coordonnées avec "
        f"d'autres contrats situés dans des rues différentes, ce qui trahit un géocodage "
        f"retombé sur un centroïde de commune ou de voie. Estimation par heuristique : la "
        f"table de géocodage expose vraisemblablement une colonne de précision, plus fiable, "
        f"qu'il suffirait d'ajouter à la requête. À vérifier avant tout contact — ces cas "
        f"sont marqués <code>geocodage_suspect</code> dans l'export CSV.")

carte_srcdoc = _html.escape(CARTE_HTML, quote=True)

detail_html = detail.reset_index().rename(columns={"feu_rattache": "Feu"})
sensi_html = sensibilite.rename(columns={
    "seuil_m": "Seuil (m)", "batis_touches": "Bâtiments", "contrats": "Contrats",
    "societaires": "Sociétaires", "retenu": ""})
synth_html = synthese[["feu", "segment", "batis_touches", "contrats", "societaires"]].rename(
    columns={"feu": "Feu", "segment": "Segment", "batis_touches": "Bâtiments",
             "contrats": "Contrats", "societaires": "Sociétaires"})

DOC = f"""<!doctype html>
<html lang="fr"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Bâtis sociétaires impactés — incendies Gironde &amp; Biscarrosse</title>
<style>{CSS}</style></head><body><div class="wrap">

<header>
  <h1>Bâtis de sociétaires impactés par les incendies</h1>
  <p>Feu de Gironde depuis le 23/07/2026, feu de Biscarrosse depuis le 24/07/2026 —
     relevés SIG au {RELEVE_SIG} · stock contrats MGAR en cours ·
     analyse du {DATE_ANALYSE}</p>
</header>
{bandeau}

<section>
  <h2><span class="n">1</span>Combien ?</h2>
  <div class="kpis">{kpi_html}</div>
</section>

<section>
  <h2><span class="n">2</span>Répartition par segment</h2>
  <div class="tbl"><table>
    <thead><tr><th class=t>Segment</th><th>Bâtiments concernés</th>
      <th>Contrats</th><th>Sociétaires</th></tr></thead>
    <tbody>{seg_rows}</tbody></table></div>
  <p class="notes" style="margin-top:.9rem">Le total en bâtiments est inférieur à la somme
  des lignes : un même immeuble peut porter plusieurs contrats de segments différents.
  {_n(KPI['hors_cible_impactes'])} contrat(s) impacté(s) relèvent d'autres segments
  (jeune, étudiant, hébergé…) et sortent du périmètre de la demande.</p>
</section>

<section>
  <h2><span class="n">3</span>Propriétaires et locataires</h2>
  <div class="tbl"><table>
    <thead><tr><th class=t>Statut d'occupation</th><th>RP</th><th>RS</th>
      <th>PNO</th><th>Total</th></tr></thead>
    <tbody>{stat_rows}</tbody></table></div>
  <p class="notes" style="margin-top:.9rem"><b>{_n(nb_proprio)} propriétaires</b> et
  <b>{_n(nb_loc)} locataires</b> parmi les contrats impactés. La distinction est lue sur
  <code>code_qualite_assure_habitation</code> : nu-propriétaire et usufruitier sont
  comptés côté propriétaire, les colocations côté locataire. Hébergé gratuit, logement
  de service et chambre en établissement ne relèvent ni de l'un ni de l'autre et
  restent à part plutôt que d'être rattachés arbitrairement.</p>
  <p class="notes">Les deux axes ne se recouvrent pas : le segment dit à quoi sert le
  logement, le statut dit qui supporte le dommage au bâti. Un PNO est propriétaire par
  construction — {"les " + _n(pno_loc) + " PNO classés « locataire » signalent une incohérence de saisie à vérifier" if pno_loc else "aucun PNO n'est ici classé « locataire », la donnée est cohérente"}.
  Le détail des modalités figure dans la colonne <code>qualite</code> de l'export CSV.</p>
</section>

<section>
  <h2><span class="n">4</span>Où ?</h2>
  <p class="notes" style="margin:-.4rem 0 .9rem">Les deux feux sont distants de 60 km :
  la vue d'ensemble situe les foyers, elle ne permet pas de lire le bâti. Utiliser
  <b>« Aller à »</b> en haut à gauche pour cadrer sur un sinistre — les emprises des
  {_n(len(batis))} bâtiments de l'emprise apparaissent en gris foncé à partir de l'échelle
  du quartier. Le sélecteur en haut à droite active ou masque chaque niveau de
  certitude. Un <b>triangle</b> signale un sinistre incendie déjà déclaré, un disque une absence de déclaration — au clic, le n° de sinistre et la date d'ouverture. Le sélecteur affiche aussi la couche <b>« Hors périmètre du feu »</b> — les contrats examinés mais non retenus — et l'<b>emprise interrogée</b>, le rectangle en pointillés qui délimite la zone extraite. La couleur porte le segment, la <b>forme</b> le statut d'occupation : disque plein pour un propriétaire, anneau pour un locataire. <b>Survoler</b> un point donne le n° de sociétaire et l'intercalaire, avec le segment et le statut ; <b>cliquer</b> ouvre la fiche complète (qualité détaillée, niveau de géocodage, adresse, distance au bâti le plus proche).</p>
  <div class="map"><iframe srcdoc="{carte_srcdoc}" loading="lazy"
    title="Carte des bâtis de sociétaires impactés"></iframe></div>
</section>

<section>
  <h2><span class="n">5</span>Détail par feu</h2>
  {_tab(synth_html)}
  <p class="notes" style="margin:1.1rem 0 .5rem">Ventilation de tous les contrats
  RP / RS / PNO par niveau de certitude :</p>
  {_tab(detail_html)}
</section>

<section>
  <h2><span class="n">6</span>Sensibilité au seuil de distance</h2>
  <p class="notes" style="margin-top:-.4rem">Le géocodage d'une adresse ne tombe pas
  toujours dans l'emprise du bâtiment. Ce tableau montre combien de contrats sont
  comptés selon la tolérance retenue — le seuil de 25 m est celui appliqué ci-dessus.</p>
  {_tab(sensi_html)}
</section>

<section>
  <h2><span class="n">7</span>Méthode et limites</h2>
  <ul class="notes">
    <li><b>Sources.</b> Emprises des bâtiments relevées par la cellule SIG
      ({_n(len(batis))} bâtiments sur les deux feux) et contours des incendies au
      {RELEVE_SIG}. Contrats issus de
      <code>contrat_mgar_gps_iris</code> × <code>contrat_mgar</code>
      (<code>tech_date_fin_historisation IS NULL</code>).</li>
    <li><b>Segmentation.</b> <code>code_sous_type</code> : 1–5 → RP, 6 → PNO, 7 → RS.
      Les segments jeune / étudiant / hébergé sont exclus du périmètre de la demande
      mais comptés séparément.</li>
    <li><b>Appariement.</b> Distances calculées en Lambert 93 (EPSG:2154) entre le point
      GPS du contrat et l'emprise du bâtiment relevé le plus proche.</li>
    <li><b>Précision du géocodage.</b> {texte_geocodage}</li>
    <li><b>Ce que la donnée ne dit pas.</b> La couche « bâti concerné » recense les
      bâtiments <i>situés dans l'emprise brûlée</i> ; elle ne qualifie pas le degré de
      destruction (totale, partielle, façade). Le chiffre est une population à
      expertiser, pas un nombre de sinistres avérés.</li>
    <li><b>Emprise minimale 50 m².</b> Les annexes plus petites (abris de jardin,
      cabanons) sont absentes de la couche source : un contrat dont seule la dépendance
      a brûlé n'est pas détecté.</li>
  </ul>
</section>

<footer>Généré le {DATE_ANALYSE} — source contrats : {MODE_SOURCE.upper()} ·
Export gestion : <code>{FICHIER_CSV.name}</code></footer>
</div></body></html>"""

FICHIER_HTML.write_text(DOC, encoding="utf-8")
print(f"✅ Livrable écrit : {FICHIER_HTML}  ({FICHIER_HTML.stat().st_size / 1024:.0f} Ko)")

---

### Pour rejouer l'analyse

1. Poste avec accès BigQuery → exécuter tel quel (`Run all`).
2. Poste sans accès → déposer l'export de la requête (cellule 3) dans
   `data_incendie/export_societaires.csv` et relancer.
3. Nouveau feu → ajouter une entrée dans `FEUX` (dossier + motifs de fichiers) ;
   le reste du notebook suit automatiquement.

Le HTML produit est autoportant hors fond de carte : mettre `FOND_DE_CARTE = None`
pour un poste sans accès Internet (les périmètres, bâtis et points restent affichés).